# FinIQ v5 — AI Financial Intelligence Platform
### 🚀 Live Data · 13 Charts · Financial Statements · NLP News · AI Insights · Peer Benchmarking · 3-Year Forecast · Grounded Q&A
**Architecture:** yfinance (data, free) · Rule-based analysis (instant) · Gemini 2.0 Flash (1 AI call) · Graham Number · Analyst Targets · Export to Excel

> **Step 1:** Fetch & Analyse (instant charts + company header)  
> **Step 2:** Load Statements (Annual / Quarterly / TTM)  
> **Step 3:** Generate AI + NLP Analysis (Gemini + news sentiment)

In [5]:
# FinIQ v5 — Install dependencies
!pip install -q "yfinance>=0.2.40" "google-genai>=1.0.0" "openpyxl" "plotly>=5.0.0" "gradio>=4.40.0" "httpx>=0.27.2" "requests"

print("✅ Done. Runtime → Restart session → Run all cells top to bottom.")


✅ Done. Runtime → Restart session → Run all cells top to bottom.


In [6]:
# Cell 2 — Imports + Gemini client
import os, json, warnings, re, time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google import genai
from google.genai import types
warnings.filterwarnings('ignore')

def get_key(name, retries=3, delay=2):
    for attempt in range(retries):
        try:
            from google.colab import userdata
            key = userdata.get(name)
            if key and key.strip():
                print(f"✅ '{name}' loaded from Colab secrets.")
                return key.strip()
            else:
                print(f"⚠️  Secret '{name}' is empty. Retrying ({attempt+1}/{retries})...")
                time.sleep(delay)
        except ImportError:
            break
        except Exception as e:
            print(f"Attempt {attempt+1}/{retries}: {type(e).__name__} — retrying in {delay}s...")
            time.sleep(delay)
    key = os.environ.get(name, "").strip()
    if key:
        print(f"✅ '{name}' loaded from environment variable.")
        return key
    print(f"\n⚠️  Could not auto-load '{name}'.")
    print("Go to 🔑 Secrets (lock icon, left sidebar) → Add secret named 'FinAIKey' → paste your Gemini API key.")
    key = input(f"Paste your Gemini API key now: ").strip()
    if key:
        os.environ[name] = key
        print(f"✅ '{name}' set manually for this session.")
        return key
    raise ValueError(f"No API key found for '{name}'.")

GEMINI_KEY = get_key('FinAIKey')
client = genai.Client(api_key=GEMINI_KEY)
AI_MODEL = "gemini-2.0-flash"
print(f"✅ Gemini client ready (model: {AI_MODEL}) — reserved for inference only")
print("Imports complete.")


✅ 'FinAIKey' loaded from Colab secrets.
✅ Gemini client ready (model: gemini-2.0-flash) — reserved for inference only
Imports complete.


In [7]:
# Cell 3 — Design system
C = dict(
    bg='#0A0E1A', panel='#0F1624', surface='#141D2E',
    border='#1E2D45', border2='#2A3F5F',
    text='#E8EEF7', muted='#6B84A3', dim='#3A4F6A',
    blue='#4D9FFF', teal='#00C896', amber='#FFB020',
    red='#FF4D6A', purple='#A78BFA', cyan='#22D3EE',
    orange='#FB923C', pink='#F472B6', lime='#84CC16',
)

def base_layout(fig, title="", height=440, subtitle=""):
    full_title = "<b>{}</b>".format(title) + ("<br><sup>{}</sup>".format(subtitle) if subtitle else "")
    fig.update_layout(
        paper_bgcolor=C["bg"], plot_bgcolor=C["panel"],
        font=dict(family="Courier New, monospace", color=C["text"], size=12),
        title=dict(text=full_title, font=dict(size=15, color=C["text"]), x=0.02),
        height=height, margin=dict(l=70, r=40, t=70, b=55),
        legend=dict(bgcolor=C["surface"], bordercolor=C["border2"], borderwidth=1,
                    font=dict(size=11), orientation="h", yanchor="bottom",
                    y=1.02, xanchor="right", x=1),
        hoverlabel=dict(bgcolor=C["surface"], bordercolor=C["border2"],
                        font=dict(color=C["text"], size=12)),
    )
    fig.update_xaxes(gridcolor=C["dim"], linecolor=C["border2"],
                     tickfont=dict(color=C["muted"], size=11), zeroline=False)
    fig.update_yaxes(gridcolor=C["dim"], linecolor=C["border2"],
                     tickfont=dict(color=C["muted"], size=11), zeroline=False)
    return fig

def wm(fig):
    fig.add_annotation(text="FinIQ v5", xref="paper", yref="paper",
        x=0.98, y=0.02, showarrow=False, font=dict(size=10, color=C["dim"]))
    return fig

print("Design system ready.")


Design system ready.


In [8]:
# Cell 4 — Ticker resolution + yfinance fetcher
import yfinance as yf
import re

# ═══════════════════════════════════════════════════════════════════
#  PURE YFINANCE FETCHER  (v4.0)
#  Zero Gemini API calls for data fetching.
#  Ticker resolution: local lookup + yfinance search → no API quota used.
#  Gemini API key is reserved ONLY for inference (insights / peer / forecast).
# ═══════════════════════════════════════════════════════════════════

# ── Built-in ticker lookup for common stocks (zero API calls) ──────────────
TICKER_MAP = {
    # Indian large-caps (NSE)
    "reliance": "RELIANCE.NS", "reliance industries": "RELIANCE.NS",
    "tcs": "TCS.NS", "tata consultancy": "TCS.NS", "tata consultancy services": "TCS.NS",
    "infosys": "INFY.NS", "infy": "INFY.NS",
    "wipro": "WIPRO.NS",
    "hdfc bank": "HDFCBANK.NS", "hdfcbank": "HDFCBANK.NS", "hdfc": "HDFCBANK.NS",
    "icici bank": "ICICIBANK.NS", "icicibank": "ICICIBANK.NS",
    "sbi": "SBIN.NS", "state bank": "SBIN.NS", "state bank of india": "SBIN.NS",
    "bajaj finance": "BAJFINANCE.NS",
    "bajaj finserv": "BAJAJFINSV.NS",
    "kotak": "KOTAKBANK.NS", "kotak mahindra": "KOTAKBANK.NS",
    "axis bank": "AXISBANK.NS",
    "hcl": "HCLTECH.NS", "hcl tech": "HCLTECH.NS", "hcl technologies": "HCLTECH.NS",
    "tech mahindra": "TECHM.NS",
    "ltimindtree": "LTIM.NS", "lti mindtree": "LTIM.NS",
    "sun pharma": "SUNPHARMA.NS", "sun pharmaceutical": "SUNPHARMA.NS",
    "dr reddy": "DRREDDY.NS", "dr reddys": "DRREDDY.NS",
    "cipla": "CIPLA.NS",
    "maruti": "MARUTI.NS", "maruti suzuki": "MARUTI.NS",
    "tata motors": "TATAMOTORS.NS",
    "mahindra": "M&M.NS", "m&m": "M&M.NS", "mahindra and mahindra": "M&M.NS",
    "titan": "TITAN.NS",
    "asian paints": "ASIANPAINT.NS",
    "nestle india": "NESTLEIND.NS",
    "hindustan unilever": "HINDUNILVR.NS", "hul": "HINDUNILVR.NS",
    "itc": "ITC.NS",
    "ongc": "ONGC.NS",
    "ntpc": "NTPC.NS",
    "power grid": "POWERGRID.NS",
    "adani enterprises": "ADANIENT.NS",
    "adani ports": "ADANIPORTS.NS",
    "adani green": "ADANIGREEN.NS",
    "jsw steel": "JSWSTEEL.NS",
    "tata steel": "TATASTEEL.NS",
    "hindalco": "HINDALCO.NS",
    "vedanta": "VEDL.NS",
    "bharti airtel": "BHARTIARTL.NS", "airtel": "BHARTIARTL.NS",
    "zomato": "ZOMATO.NS",
    "paytm": "PAYTM.NS",
    "nykaa": "FSN.NS",
    "dmart": "DMART.NS", "avenue supermarts": "DMART.NS",
    "pidilite": "PIDILITIND.NS",
    "dabur": "DABUR.NS",
    "godrej consumer": "GODREJCP.NS",
    "berger paints": "BERGEPAINT.NS",
    "havells": "HAVELLS.NS",
    "apollo hospitals": "APOLLOHOSP.NS",
    "indusind bank": "INDUSINDBK.NS",
    "yes bank": "YESBANK.NS",
    "pnb": "PNB.NS", "punjab national bank": "PNB.NS",
    "bank of baroda": "BANKBARODA.NS",
    "canara bank": "CANARABANK.NS",
    "irctc": "IRCTC.NS",
    "coal india": "COALINDIA.NS",
    "gail": "GAIL.NS",
    "bpcl": "BPCL.NS", "bharat petroleum": "BPCL.NS",
    "ioc": "IOC.NS", "indian oil": "IOC.NS",
    "hpcl": "HPCL.NS",
    # US large-caps
    "apple": "AAPL", "aapl": "AAPL",
    "microsoft": "MSFT", "msft": "MSFT",
    "google": "GOOGL", "alphabet": "GOOGL", "googl": "GOOGL",
    "amazon": "AMZN", "amzn": "AMZN",
    "meta": "META", "facebook": "META",
    "tesla": "TSLA", "tsla": "TSLA",
    "nvidia": "NVDA", "nvda": "NVDA",
    "netflix": "NFLX",
    "salesforce": "CRM",
    "adobe": "ADBE",
    "intel": "INTC",
    "amd": "AMD",
    "qualcomm": "QCOM",
    "broadcom": "AVGO",
    "paypal": "PYPL",
    "uber": "UBER",
    "airbnb": "ABNB",
    "shopify": "SHOP",
    "zoom": "ZM",
    "palantir": "PLTR",
    "coinbase": "COIN",
    "jpmorgan": "JPM", "jp morgan": "JPM",
    "goldman sachs": "GS",
    "morgan stanley": "MS",
    "bank of america": "BAC",
    "citigroup": "C", "citi": "C",
    "wells fargo": "WFC",
    "berkshire": "BRK-B",
    "johnson johnson": "JNJ", "jnj": "JNJ",
    "pfizer": "PFE",
    "moderna": "MRNA",
    "abbvie": "ABBV",
    "exxon": "XOM", "exxon mobil": "XOM",
    "chevron": "CVX",
    "walmart": "WMT",
    "costco": "COST",
    "target": "TGT",
    "home depot": "HD",
    "disney": "DIS",
    "visa": "V",
    "mastercard": "MA",
    "caterpillar": "CAT",
    "boeing": "BA",
    "samsung": "005930.KS",
}


def resolve_ticker_local(query: str) -> str:
    """
    Resolve company name to Yahoo Finance ticker using:
    1. Direct ticker check (already a valid ticker symbol)
    2. Local TICKER_MAP lookup (instant, covers 100+ common stocks)
    3. yfinance Search API (free, no Gemini needed)
    """
    query = query.strip()
    query_lower = query.lower().strip()

    # Step 1: Looks like a ticker already — validate it
    if re.match(r'^[A-Z0-9.\-^]{1,12}$', query.upper()) and len(query) <= 12:
        test = yf.Ticker(query.upper())
        info = test.info or {}
        if info.get("regularMarketPrice") or info.get("currentPrice") or info.get("longName"):
            print(f"Ticker validated directly: {query.upper()}")
            return query.upper()

    # Step 2: Local map — exact match
    if query_lower in TICKER_MAP:
        ticker = TICKER_MAP[query_lower]
        print(f"Ticker from local map: {ticker}")
        return ticker

    # Step 2b: Partial match
    for key, val in TICKER_MAP.items():
        if key in query_lower or query_lower in key:
            print(f"Ticker partial match '{key}' -> {val}")
            return val

    # Step 3: yfinance Search (free, no API quota)
    try:
        results = yf.Search(query, max_results=5).quotes
        if results:
            for r in results:
                if r.get("quoteType", "").upper() in ("EQUITY", "ETF"):
                    ticker = r.get("symbol", "")
                    if ticker:
                        print(f"Ticker from yfinance search: {ticker}")
                        return ticker
            ticker = results[0].get("symbol", "")
            if ticker:
                print(f"Ticker from yfinance search (first): {ticker}")
                return ticker
    except Exception as e:
        print(f"yfinance search failed: {e}")

    raise ValueError(
        f"Could not resolve ticker for '{query}'.\n"
        "Try entering the ticker directly (e.g. RELIANCE.NS, AAPL, TCS.NS)"
    )


# ── Helper: safely extract a row from a yfinance DataFrame ────────────────
def _yf_row(df, *keys):
    for key in keys:
        if key in df.index:
            vals = df.loc[key].tolist()
            return [round(float(v) / 1e6, 2)
                    if (v is not None and str(v) not in ("nan", "None", "<NA>")) else None
                    for v in vals]
    return [None] * len(df.columns)


# ── Main fetcher: pure yfinance, zero Gemini calls ────────────────────────
def fetch_financials_from_web(company_query: str, status_callback=None) -> dict:
    """
    Fetch financials using yfinance only. No Gemini API calls.
    Gemini is reserved exclusively for AI inference.
    """
    if status_callback:
        status_callback(f"Resolving ticker for: {company_query}...")

    ticker = resolve_ticker_local(company_query)

    if status_callback:
        status_callback(f"Fetching data from Yahoo Finance for {ticker}...")

    tk     = yf.Ticker(ticker)
    income = tk.financials
    bal    = tk.balance_sheet
    cf     = tk.cashflow
    info   = tk.info or {}

    if income is None or income.empty:
        raise ValueError(
            f"Yahoo Finance returned no data for '{ticker}'.\n"
            "Try entering the ticker directly (e.g. RELIANCE.NS, AAPL)."
        )

    income = income[income.columns[::-1]]
    bal    = bal[bal.columns[::-1]] if (bal is not None and not bal.empty) else None
    cf     = cf[cf.columns[::-1]]  if (cf  is not None and not cf.empty)  else None

    years = [int(str(c)[:4]) for c in income.columns]

    fin = {
        "Revenue":          _yf_row(income, "Total Revenue"),
        "COGS":             _yf_row(income, "Cost Of Revenue", "Cost of Goods Sold"),
        "Gross_Profit":     _yf_row(income, "Gross Profit"),
        "EBITDA":           _yf_row(income, "EBITDA", "Normalized EBITDA"),
        "EBIT":             _yf_row(income, "EBIT", "Operating Income"),
        "Net_Income":       _yf_row(income, "Net Income", "Net Income Common Stockholders"),
        "Interest_Expense": _yf_row(income, "Interest Expense"),
        "Tax_Expense":      _yf_row(income, "Tax Provision", "Income Tax Expense"),
        "Depreciation":     _yf_row(income, "Reconciled Depreciation", "Depreciation And Amortization"),
    }
    if bal is not None:
        fin.update({
            "Total_Assets":        _yf_row(bal, "Total Assets"),
            "Total_Debt":          _yf_row(bal, "Total Debt", "Long Term Debt And Capital Lease Obligation"),
            "Equity":              _yf_row(bal, "Stockholders Equity", "Common Stock Equity"),
            "Current_Assets":      _yf_row(bal, "Current Assets"),
            "Current_Liabilities": _yf_row(bal, "Current Liabilities"),
            "Retained_Earnings":   _yf_row(bal, "Retained Earnings"),
            "Receivables":         _yf_row(bal, "Accounts Receivable", "Receivables"),
            "Inventory":           _yf_row(bal, "Inventory"),
            "PPE":                 _yf_row(bal, "Net PPE", "Properties"),
        })
    if cf is not None:
        fin.update({
            "Operating_Cash_Flow": _yf_row(cf, "Operating Cash Flow", "Cash From Operations"),
            "Free_Cash_Flow":      _yf_row(cf, "Free Cash Flow"),
            "CapEx":               _yf_row(cf, "Capital Expenditure", "Purchase Of PPE"),
            "Dividends":           _yf_row(cf, "Common Stock Dividend Paid", "Dividends Paid"),
        })

    currency = info.get("currency", "USD")
    name     = info.get("longName", info.get("shortName", ticker))

    if status_callback:
        status_callback(f"Data fetched for {name} ({ticker}) — {len(years)} years | Yahoo Finance (free)")

    return {
        "company_name": name,
        "ticker":       ticker,
        "currency":     currency,
        "unit":         "millions",
        "source":       "Yahoo Finance (yfinance — free, no quota)",
        "years":        years,
        "financials":   fin,
        "info":         info,
    }


def json_to_dataframe(data: dict) -> pd.DataFrame:
    years      = data.get("years", [])
    financials = data.get("financials", {})
    rows = []
    for i, year in enumerate(years):
        row = {"Year": year}
        for metric, values in financials.items():
            row[metric] = values[i] if isinstance(values, list) and i < len(values) else None
        rows.append(row)
    df = pd.DataFrame(rows)
    for col in df.columns:
        if col != "Year":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    df = df.sort_values("Year").reset_index(drop=True)
    if "Gross_Profit" not in df.columns and {"Revenue", "COGS"}.issubset(df.columns):
        df["Gross_Profit"] = df["Revenue"] - df["COGS"]
    if "Free_Cash_Flow" not in df.columns and {"Operating_Cash_Flow", "CapEx"}.issubset(df.columns):
        df["Free_Cash_Flow"] = df["Operating_Cash_Flow"] - df["CapEx"].abs()
    if "Equity" not in df.columns and {"Total_Assets", "Total_Debt"}.issubset(df.columns):
        df["Equity"] = df["Total_Assets"] - df["Total_Debt"]
    return df


# ── Excel upload fallback ─────────────────────────────────────────────────
ALIAS = {
    "revenue": "Revenue", "sales": "Revenue", "net_sales": "Revenue", "total_revenue": "Revenue",
    "cogs": "COGS", "cost_of_goods_sold": "COGS", "cost_of_revenue": "COGS",
    "gross profit": "Gross_Profit", "gross_margin": "Gross_Profit",
    "ebitda": "EBITDA", "ebit": "EBIT", "operating_income": "EBIT",
    "net income": "Net_Income", "net_profit": "Net_Income", "profit_after_tax": "Net_Income",
    "total assets": "Total_Assets", "assets": "Total_Assets",
    "total debt": "Total_Debt", "debt": "Total_Debt", "borrowings": "Total_Debt",
    "cash": "Cash", "cash_and_equivalents": "Cash",
    "current assets": "Current_Assets", "current liabilities": "Current_Liabilities",
    "operating cash flow": "Operating_Cash_Flow", "cfo": "Operating_Cash_Flow",
    "free cash flow": "Free_Cash_Flow", "fcf": "Free_Cash_Flow",
    "retained earnings": "Retained_Earnings", "equity": "Equity",
    "shareholders_equity": "Equity", "ppe": "PPE", "fixed_assets": "PPE",
    "depreciation": "Depreciation", "receivables": "Receivables",
    "accounts_receivable": "Receivables", "inventory": "Inventory",
    "interest expense": "Interest_Expense", "tax expense": "Tax_Expense",
    "capex": "CapEx", "capital_expenditure": "CapEx",
    "dividends": "Dividends", "year": "Year", "quarter": "Quarter",
}


def load_excel_and_normalize(filepath):
    raw = pd.read_excel(filepath, header=None, engine="openpyxl")
    first_col = raw.iloc[:, 0].astype(str).str.strip().str.lower()
    year_row_idx = next((i for i, v in enumerate(first_col) if v in ("year", "metric")), None)
    if year_row_idx is not None:
        years = raw.iloc[year_row_idx, 1:].tolist()
        data = {"Year": years}
        for _, row in raw.iterrows():
            m = str(row.iloc[0]).strip().lower()
            if m in ALIAS and ALIAS[m] not in ("Year", "Quarter"):
                data[ALIAS[m]] = row.iloc[1:].tolist()
        df = pd.DataFrame(data)
    else:
        df = raw.copy()
        df.columns = raw.iloc[0]
        df = df[1:].reset_index(drop=True)
        df = df.rename(columns={c: ALIAS.get(str(c).strip().lower(), c) for c in df.columns})
    for col in df.columns:
        if col not in ["Year", "Quarter"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    if "Gross_Profit" not in df.columns and {"Revenue", "COGS"}.issubset(df.columns):
        df["Gross_Profit"] = df["Revenue"] - df["COGS"]
    if "Free_Cash_Flow" not in df.columns and {"Operating_Cash_Flow", "CapEx"}.issubset(df.columns):
        df["Free_Cash_Flow"] = df["Operating_Cash_Flow"] - df["CapEx"].abs()
    if "Equity" not in df.columns and {"Total_Assets", "Total_Debt"}.issubset(df.columns):
        df["Equity"] = df["Total_Assets"] - df["Total_Debt"]
    return df


print("Pure yfinance fetcher ready (v4.0)")
print("  Ticker resolution: local map (100+ stocks) + yfinance search — ZERO Gemini calls")
print("  Data fetch: Yahoo Finance via yfinance — free, no quota")
print("  Gemini API key: reserved exclusively for AI inference")


Pure yfinance fetcher ready (v4.0)
  Ticker resolution: local map (100+ stocks) + yfinance search — ZERO Gemini calls
  Data fetch: Yahoo Finance via yfinance — free, no quota
  Gemini API key: reserved exclusively for AI inference


In [9]:
# Cell 5 — Company header (Upgrade 1) + Ratios + Graham Number (Upgrade 5)
# ═══════════════════════════════════════════════════════════════════
# UPGRADE 1 — Company Identity Header Strip
# UPGRADE 5 — Graham Number + Analyst Targets
# ═══════════════════════════════════════════════════════════════════

def _fmt_num(val, currency=""):
    """Format large numbers with B/M/T suffix."""
    if val is None: return "—"
    try:
        v = float(val)
        sym = currency if currency else ""
        if abs(v) >= 1e12: return f"{sym}{v/1e12:.2f}T"
        if abs(v) >= 1e9:  return f"{sym}{v/1e9:.2f}B"
        if abs(v) >= 1e6:  return f"{sym}{v/1e6:.1f}M"
        return f"{sym}{v:,.2f}"
    except: return "—"

def _safe(d, *keys, default="—"):
    for k in keys:
        v = d.get(k)
        if v is not None and str(v) not in ("nan", "None", "N/A", ""):
            return v
    return default

def build_company_header(info: dict, ratios: dict) -> str:
    """Build a rich markdown company header from yfinance info dict."""
    currency = info.get("currency", "")
    sym = "₹" if currency == "INR" else ("$" if currency == "USD" else (currency + " "))

    name      = _safe(info, "longName", "shortName", default="Unknown Company")
    ticker_s  = _safe(info, "symbol", default="")
    exchange  = _safe(info, "exchange", "fullExchangeName", default="—")
    sector    = _safe(info, "sector", default="—")
    industry  = _safe(info, "industry", default="—")
    mktcap    = _fmt_num(_safe(info, "marketCap", default=None), sym)
    pe        = _safe(info, "trailingPE", default=None)
    pe_str    = f"{pe:.1f}x" if isinstance(pe, (int,float)) else "—"
    eps       = _safe(info, "trailingEps", default=None)
    eps_str   = f"{sym}{eps:.2f}" if isinstance(eps, (int,float)) else "—"
    low52     = _safe(info, "fiftyTwoWeekLow",  default=None)
    high52    = _safe(info, "fiftyTwoWeekHigh", default=None)
    curr_px   = _safe(info, "currentPrice", "regularMarketPrice", default=None)
    range_str = f"{sym}{low52:.1f} to {sym}{high52:.1f}" if isinstance(low52,(int,float)) and isinstance(high52,(int,float)) else "—"
    curr_str  = f"(Current: {sym}{curr_px:.1f})" if isinstance(curr_px,(int,float)) else ""
    beta      = _safe(info, "beta", default=None)
    beta_str  = f"{beta:.2f}" if isinstance(beta,(int,float)) else "—"
    div_yld   = _safe(info, "dividendYield", default=None)
    div_str   = f"{div_yld*100:.2f}%" if isinstance(div_yld,(int,float)) else "—"
    avg_vol   = _safe(info, "averageVolume10days", "averageVolume", default=None)
    vol_str   = _fmt_num(avg_vol) if avg_vol else "—"

    # Analyst targets — read from info dict (no extra API call, avoids 429)
    analyst_str = ""
    try:
        target_mean = info.get("targetMeanPrice") or info.get("targetMedianPrice")
        if target_mean and isinstance(curr_px, (int,float)) and curr_px > 0:
            upside = (float(target_mean) - curr_px) / curr_px * 100
            analyst_str = "\n**Analyst Target:** {}{:.1f} ({:+.1f}% upside)".format(sym, float(target_mean), upside)
        rec_key = info.get("recommendationKey", "")
        num_analysts = info.get("numberOfAnalystOpinions", "")
        if rec_key:
            rec_label = rec_key.replace("_", " ").title()
            analyst_str += " | Consensus: {} ({})".format(rec_label, num_analysts) if num_analysts else " | Consensus: {}".format(rec_label)
    except Exception:
        pass

    graham = ratios.get("Graham Number")
    graham_str = f"  **Graham Number (Intrinsic Value):** {sym}{graham:.2f}" if graham else ""

    header = (
        "---\n"
        f"## 🏢 {name} ({ticker_s})\n"
        f"**Exchange:** {exchange} | **Sector:** {sector} | **Industry:** {industry}\n"
        f"**Market Cap:** {mktcap} | **P/E (TTM):** {pe_str} | **EPS (TTM):** {eps_str}\n"
        f"**52-Week Range:** {range_str} {curr_str}\n"
        f"**Beta:** {beta_str} | **Dividend Yield:** {div_str} | **Avg Volume:** {vol_str}"
        f"{analyst_str}{graham_str}\n"
        "---"
    )
    return header


def compute_ratios(df, info=None):
    r = df.iloc[-1]
    def s(n, d, pct=False):
        try:
            v = float(n) / float(d)
            return round(v*100,2) if pct else round(v,3)
        except: return None
    ratios = {
        "Gross Margin %":    s(r.get("Gross_Profit"),        r.get("Revenue"),              pct=True),
        "EBITDA Margin %":   s(r.get("EBITDA"),              r.get("Revenue"),              pct=True),
        "Net Margin %":      s(r.get("Net_Income"),          r.get("Revenue"),              pct=True),
        "ROE %":             s(r.get("Net_Income"),          r.get("Equity"),               pct=True),
        "ROA %":             s(r.get("Net_Income"),          r.get("Total_Assets"),         pct=True),
        "Current Ratio":     s(r.get("Current_Assets"),      r.get("Current_Liabilities")),
        "Debt-to-Equity":    s(r.get("Total_Debt"),          r.get("Equity")),
        "Debt-to-Assets":    s(r.get("Total_Debt"),          r.get("Total_Assets")),
        "Interest Coverage": s(r.get("EBIT"),                r.get("Interest_Expense")),
        "Asset Turnover":    s(r.get("Revenue"),             r.get("Total_Assets")),
        "Cash Flow Quality": s(r.get("Operating_Cash_Flow"), r.get("Net_Income")),
        "FCF Margin %":      s(r.get("Free_Cash_Flow"),      r.get("Revenue"),              pct=True),
    }
    # UPGRADE 5b — Graham Number
    if info:
        try:
            shares = info.get("sharesOutstanding")
            ni_val = float(r.get("Net_Income", 0)) * 1e6
            eq_val = float(r.get("Equity", 0)) * 1e6
            if shares and shares > 0 and ni_val > 0 and eq_val > 0:
                eps_calc = ni_val / shares
                bvps     = eq_val / shares
                graham   = round((22.5 * eps_calc * bvps) ** 0.5, 2)
                ratios["Graham Number"] = graham
        except Exception:
            pass
    return ratios


def altman_z(df):
    r = df.iloc[-1]
    try:
        wc   = float(r["Current_Assets"]) - float(r["Current_Liabilities"])
        re   = float(r.get("Retained_Earnings", r.get("Net_Income", 0)))
        ebit = float(r.get("EBIT", r.get("EBITDA", r.get("Net_Income", 0))))
        eq   = float(r.get("Equity", float(r["Total_Assets"]) - float(r["Total_Debt"])))
        ta, td, rev = float(r["Total_Assets"]), float(r["Total_Debt"]), float(r["Revenue"])
        Z = round(1.2*(wc/ta)+1.4*(re/ta)+3.3*(ebit/ta)+0.6*(eq/td)+1.0*(rev/ta), 3)
        if Z > 2.99:   return Z, "Safe Zone",     C["teal"]
        elif Z > 1.81: return Z, "Grey Zone",     C["amber"]
        else:          return Z, "Distress Zone", C["red"]
    except Exception as e:
        return None, f"Insufficient data ({e})", C["muted"]

def beneish_m(df):
    if len(df) < 2: return None, "Need 2+ years", C["muted"]
    try:
        c, p = df.iloc[-1], df.iloc[-2]
        def f(col): return float(c.get(col,0)), float(p.get(col,0))
        cr,pr = f("Receivables"); cv,pv = f("Revenue"); cg,pg = f("COGS")
        ca,pa = f("Total_Assets"); cca,pca = f("Current_Assets")
        cpp,ppp = f("PPE"); cd,pd_ = f("Depreciation")
        DSRI = (cr/cv)/(pr/pv) if pr*pv else 1
        GMI  = ((pv-pg)/pv)/((cv-cg)/cv) if cv else 1
        AQI  = (1-(cca+cpp)/ca)/(1-(pca+ppp)/pa) if pa else 1
        SGI  = cv/pv if pv else 1
        DEPI = (pd_/(pd_+ppp))/(cd/(cd+cpp)) if (cd*(pd_+ppp)) else 1
        M = round(-4.84+0.92*DSRI+0.528*GMI+0.404*AQI+0.892*SGI+0.115*DEPI, 3)
        if M > -2.22: return M, "Possible Manipulation", C["red"]
        else:         return M, "Low Manipulation Risk",  C["teal"]
    except Exception as e:
        return None, f"Error: {e}", C["muted"]

print("Ratio, Risk & Company Header engines ready (v5).")


Ratio, Risk & Company Header engines ready (v5).


In [10]:
# Cell 6 — Financial Statements Engine (refined: type selector + period + row count)

STMT_ROW_LABELS = {
    "Total Revenue": "Revenue", "Cost Of Revenue": "COGS",
    "Gross Profit": "Gross Profit", "Operating Income": "EBIT (Operating Income)",
    "EBIT": "EBIT", "EBITDA": "EBITDA", "Normalized EBITDA": "EBITDA (Normalized)",
    "Net Income": "Net Income", "Net Income Common Stockholders": "Net Income (Common)",
    "Tax Provision": "Tax Expense", "Interest Expense": "Interest Expense",
    "Reconciled Depreciation": "D&A", "Diluted EPS": "EPS (Diluted)", "Basic EPS": "EPS (Basic)",
    "Total Assets": "Total Assets", "Current Assets": "Current Assets",
    "Cash And Cash Equivalents": "Cash & Equivalents",
    "Cash Cash Equivalents And Short Term Investments": "Cash & Short-Term Inv.",
    "Inventory": "Inventory", "Accounts Receivable": "Accounts Receivable",
    "Net PPE": "Net PP&E", "Goodwill": "Goodwill",
    "Total Liabilities Net Minority Interest": "Total Liabilities",
    "Current Liabilities": "Current Liabilities", "Accounts Payable": "Accounts Payable",
    "Total Debt": "Total Debt", "Long Term Debt": "Long-Term Debt",
    "Stockholders Equity": "Shareholders Equity", "Common Stock Equity": "Common Equity",
    "Retained Earnings": "Retained Earnings",
    "Operating Cash Flow": "Operating Cash Flow", "Free Cash Flow": "Free Cash Flow",
    "Capital Expenditure": "CapEx",
    "Net Income From Continuing Operations": "Net Income (Cont. Ops)",
    "Depreciation And Amortization": "D&A",
    "Change In Working Capital": "Working Capital Change",
    "Investing Cash Flow": "Investing Cash Flow", "Financing Cash Flow": "Financing Cash Flow",
    "Common Stock Dividend Paid": "Dividends Paid",
    "Repurchase Of Capital Stock": "Share Buybacks",
    "Net Issuance Payments Of Debt": "Net Debt Issuance",
    "Interest Paid Supplemental Data": "Interest Paid",
    "Income Tax Paid Supplemental Data": "Tax Paid (cash)",
}

INCOME_PRIORITY = [
    "Total Revenue","Cost Of Revenue","Gross Profit","Operating Income","EBIT","EBITDA",
    "Normalized EBITDA","Interest Expense","Tax Provision","Net Income",
    "Net Income Common Stockholders","Reconciled Depreciation","Diluted EPS","Basic EPS",
]
BALANCE_PRIORITY = [
    "Total Assets","Current Assets","Cash And Cash Equivalents",
    "Cash Cash Equivalents And Short Term Investments","Inventory","Accounts Receivable",
    "Net PPE","Goodwill","Total Liabilities Net Minority Interest","Current Liabilities",
    "Accounts Payable","Total Debt","Long Term Debt","Stockholders Equity",
    "Common Stock Equity","Retained Earnings",
]
CF_PRIORITY = [
    "Operating Cash Flow","Net Income From Continuing Operations",
    "Depreciation And Amortization","Change In Working Capital","Free Cash Flow",
    "Capital Expenditure","Investing Cash Flow","Financing Cash Flow",
    "Common Stock Dividend Paid","Repurchase Of Capital Stock",
    "Net Issuance Payments Of Debt","Interest Paid Supplemental Data",
]


def _fmt_val(v):
    try:
        v = float(v)
        if abs(v) >= 1e9:  return '{:.2f}B'.format(v / 1e9)
        if abs(v) >= 1e6:  return '{:.0f}M'.format(v / 1e6)
        if abs(v) >= 1e3:  return '{:.1f}K'.format(v / 1e3)
        return '{:,.2f}'.format(v)
    except:
        return '—'


def _yoy_delta(curr, prev):
    try:
        c, p = float(curr), float(prev)
        if abs(p) < 1e-9: return '—'
        pct = (c - p) / abs(p) * 100
        arrow = '🟢' if pct >= 0 else '🔴'
        return '{} {:+.1f}%'.format(arrow, pct)
    except:
        return '—'


def render_statement_md(raw_df, title, priority_rows, max_rows=15):
    if raw_df is None or raw_df.empty:
        return '*No data available for {}.*'.format(title)
    cols = list(raw_df.columns)
    col_labels = []
    for c in cols:
        try: col_labels.append(pd.Timestamp(c).strftime('%b %Y'))
        except: col_labels.append(str(c)[:10])
    ordered_rows = []
    seen = set()
    for key in priority_rows:
        if key in raw_df.index and key not in seen:
            ordered_rows.append(key); seen.add(key)
    for key in raw_df.index:
        if key not in seen: ordered_rows.append(key); seen.add(key)
    ordered_rows = ordered_rows[:max_rows]
    has_yoy = len(cols) >= 2
    sep_col = '|:--------|' + '|-------:|' * len(cols) + ('|------:|' if has_yoy else '')
    header  = '| **Metric** | ' + ' | '.join('**{}**'.format(c) for c in col_labels)
    header += ' | **YoY %** |' if has_yoy else ' |'
    lines = ['### {} ({} rows)\n'.format(title, len(ordered_rows)), header, sep_col]
    for key in ordered_rows:
        label = STMT_ROW_LABELS.get(key, key)
        vals  = raw_df.loc[key].tolist()
        fmt_vals = [_fmt_val(v) for v in vals]
        yoy = _yoy_delta(vals[-1], vals[-2]) if has_yoy else ''
        row = '| {} | '.format(label) + ' | '.join(fmt_vals)
        row += ' | {} |'.format(yoy) if has_yoy else ' |'
        lines.append(row)
    return '\n'.join(lines)


def fetch_statements(ticker, period):
    result = {'income': None, 'balance': None, 'cashflow': None, 'error': None}
    try:
        tk = yf.Ticker(ticker)
        if period == 'Annual':
            inc = tk.financials; bal = tk.balance_sheet; cf = tk.cashflow
        elif period == 'Quarterly':
            inc = tk.quarterly_financials; bal = tk.quarterly_balance_sheet; cf = tk.quarterly_cashflow
        else:  # TTM
            inc_q = tk.quarterly_financials
            bal_q = tk.quarterly_balance_sheet
            cf_q  = tk.quarterly_cashflow
            if inc_q is not None and not inc_q.empty and len(inc_q.columns) >= 4:
                ttm = inc_q[inc_q.columns[:4]].copy()
                flow_keys = [r for r in ttm.index if any(
                    k in r for k in ['Revenue','Income','Profit','EBIT','EBITDA','Expense','EPS']
                )]
                for row in flow_keys: ttm.loc[row] = ttm.loc[row].sum()
                inc = ttm[ttm.columns[:1]].rename(columns={ttm.columns[0]: 'TTM'})
            else: inc = inc_q
            if cf_q is not None and not cf_q.empty and len(cf_q.columns) >= 4:
                ttm_cf = cf_q[cf_q.columns[:4]].copy()
                for row in ttm_cf.index: ttm_cf.loc[row] = ttm_cf.loc[row].sum()
                cf = ttm_cf[ttm_cf.columns[:1]].rename(columns={ttm_cf.columns[0]: 'TTM'})
            else: cf = cf_q
            bal = bal_q[bal_q.columns[:1]] if (bal_q is not None and not bal_q.empty) else bal_q
        result['income'] = inc; result['balance'] = bal; result['cashflow'] = cf
    except Exception as e:
        result['error'] = str(e)
    return result


def build_statements_view(period, stmt_types, max_rows):
    max_rows = int(max_rows)
    if not _ticker or _ticker == 'UPLOAD':
        msg = '*Financial statements are available for live tickers only.*'
        return msg, msg, msg, '*—*'
    if not stmt_types:
        return ('*Select at least one statement type above.*',
                '*Select at least one statement type above.*',
                '*Select at least one statement type above.*',
                '*No statement selected.*')
    stmts = fetch_statements(_ticker, period)
    if stmts.get('error'):
        err = '*Error: {}*'.format(stmts['error'])
        return err, err, err, '*—*'
    not_selected = '*Not selected — tick the checkbox to include this statement.*'
    inc_md = render_statement_md(
        stmts.get('income'), 'Income Statement ({})'.format(period), INCOME_PRIORITY, max_rows
    ) if 'Income Statement' in stmt_types else not_selected
    bal_md = render_statement_md(
        stmts.get('balance'), 'Balance Sheet ({})'.format(period), BALANCE_PRIORITY, max_rows
    ) if 'Balance Sheet' in stmt_types else not_selected
    cf_md = render_statement_md(
        stmts.get('cashflow'), 'Cash Flow Statement ({})'.format(period), CF_PRIORITY, max_rows
    ) if 'Cash Flow' in stmt_types else not_selected
    n = len(stmt_types)
    status = '✅ Loaded {} statement{} | Period: {} | Max rows: {}'.format(
        n, 's' if n != 1 else '', period, max_rows
    )
    return inc_md, bal_md, cf_md, status


print('Financial Statements engine ready — type selector + period + row-count slider.')


Financial Statements engine ready — type selector + period + row-count slider.


In [11]:
# Cell 7 — 13 Chart functions (10 original + EPS, Waterfall, Price History — Upgrade 4)
def get_years(df):
    return df["Year"].astype(str).tolist() if "Year" in df.columns else df.index.astype(str).tolist()

def viz_revenue(df):
    yrs = get_years(df)
    fig = make_subplots(specs=[[{"secondary_y":True}]])
    for col, name, color, op in [
        ("Revenue","Revenue",C["blue"],0.9),("Gross_Profit","Gross Profit",C["teal"],0.9),
        ("Net_Income","Net Income",C["purple"],0.9),("EBITDA","EBITDA",C["cyan"],0.7),
    ]:
        if col in df.columns:
            fig.add_trace(go.Bar(x=yrs, y=df[col].round(2), name=name,
                marker_color=color, marker_line_width=0, opacity=op,
                hovertemplate=f"<b>{name}</b><br>%{{y:,.0f}}<extra></extra>"), secondary_y=False)
    if "Revenue" in df.columns and len(df) > 1:
        g = (df["Revenue"].pct_change()*100).round(1)
        fig.add_trace(go.Scatter(x=yrs, y=g, name="YoY Growth %", mode="lines+markers+text",
            line=dict(color=C["amber"],width=2,dash="dot"),
            marker=dict(size=8,symbol="diamond",color=C["amber"]),
            text=[f"{v:+.1f}%" if pd.notna(v) else "" for v in g],
            textposition="top center",textfont=dict(size=10,color=C["amber"]),
            hovertemplate="<b>YoY Growth</b><br>%{y:.1f}%<extra></extra>"), secondary_y=True)
    fig.update_yaxes(title_text="<b>Amount</b>",secondary_y=False,gridcolor=C["dim"])
    fig.update_yaxes(title_text="<b>Growth %</b>",secondary_y=True,gridcolor="rgba(0,0,0,0)")
    fig.update_layout(barmode="group")
    base_layout(fig,"Revenue and Profit Waterfall",480,"Revenue, Gross Profit, Net Income with YoY growth overlay")
    return wm(fig)

def hex_to_rgba(hex_color, alpha=0.09):
    h = hex_color.lstrip('#')
    r,g,b = int(h[0:2],16),int(h[2:4],16),int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

def viz_margins(df):
    yrs = get_years(df)
    fig = go.Figure()
    if "Revenue" not in df.columns: return fig
    defs = [
        ("Gross_Profit","Gross Margin %",C["teal"],30),
        ("EBITDA","EBITDA Margin %",C["cyan"],15),
        ("Net_Income","Net Margin %",C["purple"],5),
        ("Free_Cash_Flow","FCF Margin %",C["amber"],8),
        ("Operating_Cash_Flow","OCF Margin %",C["lime"],10),
    ]
    for col,name,color,thresh in defs:
        if col in df.columns:
            margin = (df[col]/df["Revenue"]*100).round(2)
            fig.add_trace(go.Scatter(x=yrs,y=margin,name=name,mode="lines+markers",
                line=dict(color=color,width=2),marker=dict(size=7,color=color),
                fill="tozeroy",fillcolor=hex_to_rgba(color,0.07),
                hovertemplate=f"<b>{name}</b><br>%{{y:.1f}}%<extra></extra>"))
    base_layout(fig,"Margin Trends",440,"All margins as % of Revenue — trend and level analysis")
    return wm(fig)

def viz_balance_sheet(df):
    yrs = get_years(df)
    fig = go.Figure()
    for col,name,color in [("Total_Assets","Total Assets",C["blue"]),
        ("Equity","Equity",C["teal"]),("Total_Debt","Total Debt",C["red"]),
        ("Current_Assets","Current Assets",C["cyan"])]:
        if col in df.columns:
            fig.add_trace(go.Bar(x=yrs,y=df[col],name=name,marker_color=color,
                marker_line_width=0,opacity=0.85,
                hovertemplate=f"<b>{name}</b><br>%{{y:,.0f}}<extra></extra>"))
    fig.update_layout(barmode="group")
    base_layout(fig,"Balance Sheet Structure",440,"Assets, Equity, Debt and Current Assets trends")
    return wm(fig)

def viz_cashflow(df):
    yrs = get_years(df)
    fig = go.Figure()
    for col,name,color in [("Operating_Cash_Flow","Operating CF",C["teal"]),
        ("Free_Cash_Flow","Free CF",C["blue"]),("CapEx","CapEx",C["red"]),
        ("Net_Income","Net Income",C["purple"])]:
        if col in df.columns:
            vals = df[col].round(2)
            fig.add_trace(go.Bar(x=yrs,y=vals,name=name,marker_color=color,
                marker_line_width=0,opacity=0.85,
                hovertemplate=f"<b>{name}</b><br>%{{y:,.0f}}<extra></extra>"))
    fig.update_layout(barmode="group")
    base_layout(fig,"Cash Flow Analysis",440,"Operating CF, FCF, CapEx vs Net Income")
    return wm(fig)

def viz_ratio_gauges(ratios):
    keys = ["Gross Margin %","Net Margin %","ROE %","ROA %","Current Ratio","Debt-to-Equity"]
    vals = [ratios.get(k) for k in keys]
    ranges = [(0,80),(0,30),(0,40),(0,20),(0,3),(0,3)]
    fig = make_subplots(rows=2,cols=3,specs=[[{"type":"indicator"}]*3,[{"type":"indicator"}]*3])
    for i,(k,v,rng) in enumerate(zip(keys,vals,ranges)):
        r,c = divmod(i,3)
        fig.add_trace(go.Indicator(
            mode="gauge+number",value=v,title={"text":k,"font":{"size":11,"color":C["muted"]}},
            gauge={"axis":{"range":rng,"tickcolor":C["muted"]},
                   "bar":{"color":C["blue"]},
                   "bgcolor":C["panel"],"bordercolor":C["border2"],
                   "steps":[{"range":[rng[0],rng[1]*0.4],"color":C["surface"]},
                             {"range":[rng[1]*0.4,rng[1]*0.7],"color":C["dim"]}],
                   "threshold":{"line":{"color":C["amber"],"width":3},"thickness":0.8,"value":v or 0}},
            number={"font":{"color":C["text"],"size":16}}), row=r+1,col=c+1)
    base_layout(fig,"Key Ratio Gauges",480,"Latest-year ratio snapshot across 6 dimensions")
    return wm(fig)

def viz_altman_gauge(Z, zone, zcol):
    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=Z if Z else 0,
        title={"text":f"Altman Z-Score<br><sup>{zone}</sup>","font":{"color":C["text"],"size":14}},
        delta={"reference":1.81,"increasing":{"color":C["teal"]},"decreasing":{"color":C["red"]}},
        gauge={
            "axis":{"range":[0,5],"tickvals":[0,1.81,2.99,5],"ticktext":["0","1.81","2.99","5"],
                    "tickcolor":C["muted"]},
            "bar":{"color":zcol,"thickness":0.3},
            "bgcolor":C["panel"],"bordercolor":C["border2"],
            "steps":[{"range":[0,1.81],"color":"rgba(255,77,106,0.13)"},
                     {"range":[1.81,2.99],"color":"rgba(255,176,32,0.13)"},
                     {"range":[2.99,5],"color":"rgba(0,200,150,0.13)"}],
            "threshold":{"line":{"color":"white","width":3},"thickness":0.8,"value":Z or 0}
        },
        number={"font":{"color":zcol,"size":32},"suffix":""}))
    base_layout(fig,"Altman Z-Score — Bankruptcy Risk",380,"< 1.81 Distress | 1.81–2.99 Grey | > 2.99 Safe")
    return wm(fig)

def viz_beneish(df):
    if len(df) < 2: return None
    yrs = get_years(df)
    fig = go.Figure()
    metrics = ["Revenue","Receivables","COGS","Total_Assets","PPE","Depreciation"]
    colors  = [C["blue"],C["teal"],C["red"],C["amber"],C["cyan"],C["purple"]]
    for col,color in zip(metrics,colors):
        if col in df.columns:
            pct = (df[col].pct_change()*100).round(1)
            fig.add_trace(go.Scatter(x=yrs,y=pct,name=col,mode="lines+markers",
                line=dict(color=color,width=2),marker=dict(size=7),
                hovertemplate=f"<b>{col} YoY %</b><br>%{{y:.1f}}%<extra></extra>"))
    fig.add_hline(y=0,line_dash="dash",line_color=C["muted"],opacity=0.5)
    base_layout(fig,"Beneish M-Score Components",440,"YoY % changes in key Beneish drivers")
    return wm(fig)

def viz_dupont(df):
    if "Net_Income" not in df.columns: return None
    yrs = get_years(df)
    fig = go.Figure()
    if all(c in df.columns for c in ["Net_Income","Revenue"]):
        npm = (df["Net_Income"]/df["Revenue"]*100).round(2)
        fig.add_trace(go.Scatter(x=yrs,y=npm,name="Net Margin %",mode="lines+markers",
            line=dict(color=C["teal"],width=2),marker=dict(size=8)))
    if all(c in df.columns for c in ["Revenue","Total_Assets"]):
        at = (df["Revenue"]/df["Total_Assets"]).round(3)
        fig.add_trace(go.Scatter(x=yrs,y=at,name="Asset Turnover",mode="lines+markers",
            line=dict(color=C["blue"],width=2),marker=dict(size=8)))
    if all(c in df.columns for c in ["Total_Assets","Equity"]):
        lev = (df["Total_Assets"]/df["Equity"]).round(3)
        fig.add_trace(go.Scatter(x=yrs,y=lev,name="Equity Multiplier",mode="lines+markers",
            line=dict(color=C["amber"],width=2),marker=dict(size=8)))
    if all(c in df.columns for c in ["Net_Income","Equity"]):
        roe = (df["Net_Income"]/df["Equity"]*100).round(2)
        fig.add_trace(go.Scatter(x=yrs,y=roe,name="ROE % (DuPont)",mode="lines+markers",
            line=dict(color=C["purple"],width=2,dash="dot"),marker=dict(size=9,symbol="diamond")))
    base_layout(fig,"DuPont ROE Decomposition",440,"Net Margin × Asset Turnover × Equity Multiplier = ROE")
    return wm(fig)

def viz_dcf_heatmap(df):
    if "Free_Cash_Flow" not in df.columns: return None
    last_fcf = df["Free_Cash_Flow"].dropna().iloc[-1] if len(df["Free_Cash_Flow"].dropna()) > 0 else None
    if last_fcf is None or last_fcf <= 0: return None
    growth_rates = [0.03,0.05,0.07,0.10,0.12,0.15]
    disc_rates   = [0.06,0.08,0.10,0.12,0.14,0.16]
    z = []
    for g in growth_rates:
        row_vals = []
        for d in disc_rates:
            if d <= g: row_vals.append(None)
            else:
                tv = last_fcf*(1+g)/(d-g)
                pv = sum([last_fcf*(1+g)**t/(1+d)**t for t in range(1,6)])
                row_vals.append(round((pv+tv)/(last_fcf*10),2) if last_fcf*10 != 0 else None)
        z.append(row_vals)
    fig = go.Figure(go.Heatmap(
        z=z, x=[f"{int(d*100)}%" for d in disc_rates],
        y=[f"{int(g*100)}%" for g in growth_rates],
        colorscale=[[0,C["red"]],[0.5,C["amber"]],[1,C["teal"]]],
        text=[[str(v) if v else "N/A" for v in row] for row in z],
        texttemplate="%{text}x", hovertemplate="Growth: %{y}<br>Discount: %{x}<br>EV/FCF: %{z}x<extra></extra>"))
    base_layout(fig,"DCF Sensitivity Heatmap",440,"EV/FCF multiple across growth rate vs discount rate scenarios")
    return wm(fig)

def viz_red_flags(df, ratios, Z, zone, M, m_lbl):
    flags = []  # list of (label, severity)
    if ratios.get("Net Margin %") is not None and ratios["Net Margin %"] < 3:
        flags.append(("Net Margin < 3%", "HIGH"))
    if ratios.get("Current Ratio") is not None and ratios["Current Ratio"] < 1.0:
        flags.append(("Current Ratio < 1", "HIGH"))
    if ratios.get("Debt-to-Equity") is not None and ratios["Debt-to-Equity"] > 2.0:
        flags.append(("D/E Ratio > 2x", "HIGH"))
    if Z is not None and Z < 1.81:
        flags.append(("Altman Z < 1.81 — Distress Zone", "HIGH"))
    if M is not None and M > -2.22:
        flags.append(("Beneish M > -2.22 — Manipulation Risk", "HIGH"))
    if "Revenue" in df.columns and len(df) > 2:
        recent = df["Revenue"].pct_change().tail(2).mean()
        if recent < 0:
            flags.append(("Revenue Declining", "MEDIUM"))
    if ratios.get("Cash Flow Quality") is not None and ratios["Cash Flow Quality"] < 0.5:
        flags.append(("Low Cash Flow Quality", "MEDIUM"))
    if not flags:
        flags = [("No Major Red Flags Detected", "LOW")]
    colors_map = {"HIGH":C["red"],"MEDIUM":C["amber"],"LOW":C["teal"]}
    fig = go.Figure()
    for i,(lbl,sev) in enumerate(flags):
        fig.add_trace(go.Bar(x=[1],y=[1],orientation="v",showlegend=False,
            marker_color=colors_map.get(sev,C["muted"]),
            hoverinfo="text",hovertext=f"<b>{lbl}</b><br>Severity: {sev}"))
    labels_ = [f"[{sev}] {lbl}" for lbl,sev in flags]
    sevs    = [sev for _,sev in flags]
    clrs    = [colors_map.get(s,C["muted"]) for s in sevs]
    fig = go.Figure(go.Bar(y=labels_,x=[3 if s=="HIGH" else 2 if s=="MEDIUM" else 1 for s in sevs],
        orientation="h",marker_color=clrs,marker_line_width=0,
        text=sevs,textposition="inside",textfont=dict(size=11,color=C["text"])))
    base_layout(fig,"Red Flag Scorecard",max(350,len(flags)*55+100),"Detected financial red flags by severity")
    return wm(fig)



# UPGRADE 4a — EPS Trend
def viz_eps_trend(df, info=None):
    yrs = get_years(df)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    eps_vals = None
    if info and "sharesOutstanding" in info and "Net_Income" in df.columns:
        try:
            shares = float(info["sharesOutstanding"])
            if shares > 0: eps_vals = (df["Net_Income"] * 1e6 / shares).round(2)
        except: pass
    if eps_vals is not None and not eps_vals.isna().all():
        fig.add_trace(go.Bar(x=yrs, y=eps_vals, name="EPS (computed)",
            marker_color=C["purple"], marker_line_width=0, opacity=0.85,
            hovertemplate="<b>EPS</b><br>%{y:.2f}<extra></extra>"), secondary_y=False)
    elif "Net_Income" in df.columns:
        fig.add_trace(go.Bar(x=yrs, y=df["Net_Income"].round(2), name="Net Income (M)",
            marker_color=C["purple"], marker_line_width=0, opacity=0.85,
            hovertemplate="<b>Net Income</b><br>%{y:,.0f}M<extra></extra>"), secondary_y=False)
    if info:
        trailing_pe = info.get("trailingPE")
        if trailing_pe and isinstance(trailing_pe, (int, float)) and not pd.isna(trailing_pe):
            fig.add_hline(y=trailing_pe, line_dash="dot", line_color=C["amber"],
                annotation_text="Current P/E: {:.1f}x".format(trailing_pe),
                annotation_font_color=C["amber"])
    fig.update_yaxes(title_text="<b>EPS / Net Income</b>", secondary_y=False, gridcolor=C["dim"])
    fig.update_yaxes(title_text="<b>P/E Ratio</b>", secondary_y=True, gridcolor="rgba(0,0,0,0)")
    base_layout(fig, "EPS Trend & P/E Analysis", 440, "Earnings per share with P/E reference")
    return wm(fig)

# UPGRADE 4b — Revenue Waterfall
def viz_waterfall(df):
    if "Revenue" not in df.columns: return None
    r = df.iloc[-1]
    year_label = str(r.get("Year", "Latest"))
    items, values, measures = [], [], []
    rev = r.get("Revenue")
    if rev is None: return None
    items.append("Revenue"); values.append(round(float(rev),1)); measures.append("absolute")
    if r.get("COGS") is not None:
        cogs = round(float(r["COGS"]),1)
        items.append("COGS"); values.append(-abs(cogs)); measures.append("relative")
    gp = r.get("Gross_Profit")
    if gp is not None:
        items.append("Gross Profit"); values.append(round(float(gp),1)); measures.append("total")
    ebit = r.get("EBIT")
    if ebit is not None and gp is not None:
        opex = round(float(gp) - float(ebit), 1)
        items.append("OpEx & D&A"); values.append(-abs(opex)); measures.append("relative")
    ni = r.get("Net_Income")
    if ni is not None:
        items.append("Net Income"); values.append(round(float(ni),1)); measures.append("total")
    if len(items) < 3: return None
    fig = go.Figure(go.Waterfall(
        name=year_label, orientation="v", measure=measures, x=items, y=values,
        connector={"line": {"color": C["border2"], "width": 1}},
        increasing={"marker": {"color": C["teal"]}},
        decreasing={"marker": {"color": C["red"]}},
        totals={"marker": {"color": C["blue"]}},
        hovertemplate="<b>%{x}</b><br>%{y:,.0f}M<extra></extra>"
    ))
    base_layout(fig, "Revenue Bridge — {}".format(year_label), 440,
                "Revenue to Gross Profit to Net Income breakdown")
    return wm(fig)

# UPGRADE 4c — Price History
def viz_price_history(ticker):
    try:
        hist = yf.Ticker(ticker).history(period="1y")
        if hist is None or hist.empty: return None
    except Exception as e:
        print("Price history error: {}".format(e)); return None
    hist.index = pd.to_datetime(hist.index)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        vertical_spacing=0.06, row_heights=[0.7, 0.3])
    fig.add_trace(go.Candlestick(
        x=hist.index, open=hist["Open"], high=hist["High"],
        low=hist["Low"], close=hist["Close"], name="OHLC",
        increasing_line_color=C["teal"], decreasing_line_color=C["red"],
        increasing_fillcolor=C["teal"],  decreasing_fillcolor=C["red"],
    ), row=1, col=1)
    for window, color, nm in [(50, C["amber"], "SMA 50"), (200, C["blue"], "SMA 200")]:
        if len(hist) >= window:
            sma = hist["Close"].rolling(window).mean()
            fig.add_trace(go.Scatter(x=hist.index, y=sma, name=nm,
                line=dict(color=color, width=1.5, dash="dot")), row=1, col=1)
    vol_colors = [C["teal"] if c >= o else C["red"]
                  for c, o in zip(hist["Close"], hist["Open"])]
    fig.add_trace(go.Bar(x=hist.index, y=hist["Volume"], name="Volume",
        marker_color=vol_colors, marker_line_width=0, opacity=0.7), row=2, col=1)
    fig.update_layout(xaxis_rangeslider_visible=False)
    base_layout(fig, "Price History — {} (1Y)".format(ticker), 520,
                "OHLC Candlestick with SMA 50/200 and Volume")
    return wm(fig)

print("All 13 chart functions ready: 10 original + EPS Trend + Revenue Waterfall + Price History.")


All 13 chart functions ready: 10 original + EPS Trend + Revenue Waterfall + Price History.


In [12]:
# Cell 8 — AI Engine v5 (enhanced prompt, news cards, topic tags — Upgrades 3 & 8)
import time as _time
import threading
import re as _re

_news_tags_global = ""  # Upgrade 8

# ════════════════════════════════════════════════════════════════════
#  AI INFERENCE ENGINE  (v5.0)
#  Strategy: Gemini runs in a background thread with 25s hard deadline.
#            If it doesn't respond → instant rule-based fallback text.
#            Charts/ratios NEVER wait for AI.
#  NEW: NLP news sentiment analysis via yfinance news feed (no extra API).
# ════════════════════════════════════════════════════════════════════

# ── Gemini call: fire-and-forget with hard deadline ───────────────
def _gemini_call(prompt: str, timeout: int = 25) -> str:
    """
    Calls Gemini in a thread. If no response within `timeout` seconds,
    returns None immediately — caller uses rule-based fallback instead.
    """
    result = [None]
    error  = [None]

    def _worker():
        try:
            resp = client.models.generate_content(
                model=AI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            result[0] = resp.text
        except Exception as e:
            error[0] = str(e)

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join(timeout=timeout)

    if t.is_alive():
        print(f"Gemini timeout after {timeout}s — using rule-based analysis")
        return None
    if error[0]:
        err = error[0].lower()
        if any(k in err for k in ["429", "quota", "resource_exhausted", "too many"]):
            print(f"Rate limit: {error[0][:80]}")
        else:
            print(f"Gemini error: {error[0][:80]}")
        return None
    return result[0]


# ── Rule-based insight generator (instant, no API) ───────────────
def _rule_based_insights(df, ratios, Z, zone, M, m_lbl) -> str:
    lines = ["## 📊 Financial Analysis (Rule-Based)\n"]

    # Revenue trend
    if "Revenue" in df.columns and len(df) >= 2:
        rev = df["Revenue"].dropna()
        if len(rev) >= 2:
            cagr = ((rev.iloc[-1] / rev.iloc[0]) ** (1 / max(len(rev)-1, 1)) - 1) * 100
            trend = "growing" if cagr > 5 else ("declining" if cagr < -2 else "stable")
            lines.append(f"**Revenue:** {trend} at {cagr:.1f}% CAGR over {len(rev)} years "
                         f"(₹{rev.iloc[-1]:,.0f}M latest)")

    # Profitability
    nm = ratios.get("Net Margin %")
    gm = ratios.get("Gross Margin %")
    if nm is not None:
        quality = "strong" if nm > 15 else ("acceptable" if nm > 5 else "thin")
        lines.append(f"**Profitability:** Net margin {nm:.1f}% ({quality})"
                     + (f", Gross margin {gm:.1f}%" if gm else ""))

    # Leverage
    de = ratios.get("Debt-to-Equity")
    cr = ratios.get("Current Ratio")
    if de is not None:
        lev = "conservative" if de < 0.5 else ("moderate" if de < 1.5 else "high")
        lines.append(f"**Leverage:** D/E {de:.2f}x ({lev})"
                     + (f", Current ratio {cr:.2f}x" if cr else ""))

    # Returns
    roe = ratios.get("ROE %")
    roa = ratios.get("ROA %")
    if roe is not None:
        lines.append(f"**Returns:** ROE {roe:.1f}%" + (f", ROA {roa:.1f}%" if roa else ""))

    # Cash flow
    cfq = ratios.get("Cash Flow Quality")
    fcf = ratios.get("FCF Margin %")
    if cfq is not None:
        quality = "high" if cfq > 1.0 else ("moderate" if cfq > 0.6 else "low")
        lines.append(f"**Cash Flow Quality:** {cfq:.2f}x ({quality})"
                     + (f", FCF margin {fcf:.1f}%" if fcf else ""))

    # Risk scores
    if Z is not None:
        lines.append(f"**Altman Z-Score:** {Z} → {zone}")
    if M is not None:
        lines.append(f"**Beneish M-Score:** {M} → {m_lbl}")

    # Recommendation
    score = 0
    if nm and nm > 10: score += 1
    if de and de < 1.0: score += 1
    if cfq and cfq > 0.8: score += 1
    if Z and Z > 2.99: score += 1
    if roe and roe > 12: score += 1
    rec = "**BUY**" if score >= 4 else ("**HOLD**" if score >= 2 else "**REVIEW**")
    lines.append(f"\n### Recommendation: {rec} (score {score}/5)")
    lines.append("\n*Note: Rule-based analysis. Click **Generate AI Insights** for Gemini-powered analysis.*")
    return "\n\n".join(lines)


def _rule_based_peer(ratios) -> str:
    benchmarks = {
        "Gross Margin %":    ("~40-60% (tech), ~20-35% (mfg)", 40),
        "Net Margin %":      ("~10-20% (tech), ~5-10% (mfg)", 10),
        "ROE %":             ("~15-25% healthy", 15),
        "Current Ratio":     ("~1.5-2.5x healthy", 1.5),
        "Debt-to-Equity":    ("< 1.0x conservative", 1.0),
        "Cash Flow Quality": ("> 1.0x strong", 1.0),
    }
    rows = ["| Metric | Company | Benchmark | Status |",
            "|--------|---------|-----------|--------|"]
    for metric, (bench, threshold) in benchmarks.items():
        val = ratios.get(metric)
        if val is None:
            rows.append(f"| {metric} | N/A | {bench} | — |")
            continue
        if "Debt" in metric:
            status = "✅ Good" if val < threshold else "⚠️ High"
        else:
            status = "✅ Good" if val >= threshold else "⚠️ Below avg"
        rows.append(f"| {metric} | {val} | {bench} | {status} |")
    rows.append("\n*Rule-based benchmarks. Click **Generate AI Insights** for Gemini peer analysis.*")
    return "\n".join(rows)


def _rule_based_forecast(df) -> str:
    if "Revenue" not in df.columns or len(df) < 2:
        return "Insufficient data for forecast."
    rev = df["Revenue"].dropna()
    ni  = df["Net_Income"].dropna() if "Net_Income" in df.columns else None
    cagr = ((rev.iloc[-1] / rev.iloc[0]) ** (1 / max(len(rev)-1, 1)) - 1)
    last_rev = rev.iloc[-1]
    lines = ["## 📈 3-Year Forward Forecast (Trend Extrapolation)\n",
             f"Base CAGR: {cagr*100:.1f}% | Latest Revenue: {last_rev:,.0f}M\n",
             "| Year | Revenue (M) | Growth |",
             "|------|------------|--------|"]
    for yr in range(1, 4):
        proj = last_rev * (1 + cagr) ** yr
        lines.append(f"| +{yr} | {proj:,.0f} | {cagr*100:+.1f}% |")
    if ni is not None and len(ni) > 0:
        nm = ni.iloc[-1] / rev.iloc[-1]
        lines.append(f"\nAssumed Net Margin: {nm*100:.1f}% → "
                     f"Projected Net Income Y+1: {last_rev*(1+cagr)*nm:,.0f}M")
    bear_cagr = cagr * 0.5
    bull_cagr = min(cagr * 1.5, 0.35)
    lines.append("\n### Scenario Analysis")
    lines.append("| Scenario | CAGR | Revenue Y+3 |")
    lines.append("|----------|------|------------|")
    lines.append("| Bear | {:+.1f}% | {:,.0f}M |".format(bear_cagr*100, last_rev*(1+bear_cagr)**3))
    lines.append("| Base | {:+.1f}% | {:,.0f}M |".format(cagr*100, last_rev*(1+cagr)**3))
    lines.append("| Bull | {:+.1f}% | {:,.0f}M |".format(bull_cagr*100, last_rev*(1+bull_cagr)**3))
    lines.append("\n*Statistical extrapolation. Click **Generate AI Insights** for Gemini forecast.*")
    return "\n".join(lines)


# ── NLP News Sentiment (yfinance — no extra API needed) ──────────
POSITIVE_WORDS = {
    "growth", "profit", "record", "strong", "beat", "exceed", "surge",
    "rally", "upgrade", "buy", "outperform", "dividend", "acquisition",
    "expansion", "revenue", "gain", "rise", "increase", "robust", "positive",
    "innovation", "launch", "partnership", "award", "milestone", "improve"
}
NEGATIVE_WORDS = {
    "loss", "decline", "miss", "weak", "cut", "downgrade", "sell", "underperform",
    "lawsuit", "fraud", "penalty", "fine", "recall", "resign", "layoff",
    "debt", "default", "crisis", "fall", "drop", "concern", "risk",
    "investigation", "probe", "warning", "downside", "delay", "reduce"
}

def analyze_news_sentiment(ticker: str, max_articles: int = 15) -> dict:
    """
    Fetch recent news from yfinance and run rule-based NLP sentiment.
    Returns dict with score, label, articles, and summary text.
    """
    try:
        import yfinance as yf
        tk = yf.Ticker(ticker)
        news = tk.news or []
    except Exception as e:
        return {"error": str(e), "articles": [], "score": 0, "label": "No data"}

    articles = news[:max_articles]
    if not articles:
        return {"error": "No news found", "articles": [], "score": 0, "label": "No data"}

    scored = []
    total_pos = total_neg = 0

    for item in articles:
        # yfinance news structure varies — handle both old and new formats
        content = item.get("content", {})
        title = (content.get("title") or item.get("title") or "").lower()
        summary = (content.get("summary") or item.get("summary") or
                   content.get("description") or "").lower()
        text = title + " " + summary
        words = set(_re.findall(r'\b\w+\b', text))

        pos = len(words & POSITIVE_WORDS)
        neg = len(words & NEGATIVE_WORDS)
        total_pos += pos
        total_neg += neg

        score = pos - neg
        sentiment = "Positive" if score > 0 else ("Negative" if score < 0 else "Neutral")

        # Get display title
        display_title = (content.get("title") or item.get("title") or "No title")[:80]
        pub_date = (content.get("pubDate") or item.get("providerPublishTime") or "")
        if isinstance(pub_date, int):
            from datetime import datetime
            pub_date = datetime.fromtimestamp(pub_date).strftime("%Y-%m-%d")

        url_val = ""
        try: url_val = (content.get("canonicalUrl", {}) or {}).get("url", "") or item.get("link", "") or ""
        except: pass
        scored.append({
            "title":     display_title,
            "sentiment": sentiment,
            "score":     score,
            "date":      str(pub_date)[:10],
            "pos_words": pos,
            "neg_words": neg,
            "url":       url_val,
            "summary":   (content.get("summary") or item.get("summary") or "")[:150],
        })

    net = total_pos - total_neg
    n   = len(scored)
    pos_count = sum(1 for s in scored if s["sentiment"] == "Positive")
    neg_count = sum(1 for s in scored if s["sentiment"] == "Negative")
    neu_count = n - pos_count - neg_count

    overall_label = ("Bullish 📈" if net > 3 else
                     "Bearish 📉" if net < -3 else "Neutral ➡️")

    summary_md = "## News Sentiment Analysis ({} articles)\n\n".format(n)
    summary_md += "**Overall Sentiment: {}**\n".format(overall_label)
    summary_md += "Positive: {} | Negative: {} | Neutral: {} | Net: {:+d}\n\n".format(pos_count, neg_count, neu_count, net)
    summary_md += "### Recent Headlines\n| Date | Headline | Sentiment |\n|------|----------|-----------|\n"
    for s in scored[:10]:
        em2 = "[+]" if s["sentiment"]=="Positive" else ("-" if s["sentiment"]=="Negative" else "~")
        summary_md += "| {} | {}... | {} {} |\n".format(s["date"], s["title"][:60], em2, s["sentiment"])

    # UPGRADE 3a — rich news cards
    cards_md = "## Latest News\n\n"
    for s in scored[:8]:
        em = "[+]" if s["sentiment"]=="Positive" else ("-" if s["sentiment"]=="Negative" else "~")
        url_s = s.get("url", "")
        title_p = "[{}]({})".format(s["title"][:70], url_s) if url_s else s["title"][:70]
        cards_md += "### {} {}\n".format(em, title_p)
        cards_md += "**Date:** {} | **Sentiment:** {} | **Score:** {:+d}\n".format(s["date"], s["sentiment"], s["score"])
        if s.get("summary"): cards_md += "> {}...\n".format(s["summary"])
        cards_md += "---\n"
    return {
        "score": net, "label": overall_label,
        "pos_count": pos_count, "neg_count": neg_count, "neu_count": neu_count,
        "articles": scored, "summary_md": summary_md, "cards_md": cards_md,
    }


def run_single_ai_call(df, ratios, Z, zone, M, m_lbl, news_data: dict, company_name="", ticker="", currency="", unit="millions") -> tuple:
    """
    ONE Gemini call that returns insights + peer comparison + forecast together.
    Parsed into 3 sections. Falls back to rule-based instantly on any failure.
    """
    cols = [c for c in ["Year","Revenue","Gross_Profit","EBITDA","Net_Income",
                        "Total_Assets","Total_Debt","Equity","Operating_Cash_Flow"] if c in df.columns]
    ctx = df[cols].to_string(index=False)
    ratio_str = "\n".join([f"- {k}: {v}" for k,v in ratios.items() if v is not None])
    news_label = news_data.get("label","N/A")
    news_score = news_data.get("score", 0)
    pos_c = news_data.get("pos_count",0)
    neg_c = news_data.get("neg_count",0)

    # Get last revenue and cagr for forecast fallback
    rev = df["Revenue"].dropna() if "Revenue" in df.columns else None
    cagr_str = ""
    if rev is not None and len(rev) >= 2:
        cagr = ((rev.iloc[-1]/rev.iloc[0])**(1/max(len(rev)-1,1))-1)*100
        cagr_str = f"Historical Revenue CAGR: {cagr:.1f}%"

    graham = ratios.get("Graham Number")
    graham_str = "Graham Number = {:.2f}".format(graham) if graham else ""

    prompt = (
        "You are a senior equity analyst at a top-tier investment bank.\n"
        "Analyze the data and write THREE sections. Use EXACT markers. Each section max 300 words.\n\n"
        "Company: {} ({}) | Currency: {} | Unit: {}\n".format(company_name, ticker, currency, unit) +
        "Financial Data:\n" + ctx + "\n\nKey Ratios:\n" + ratio_str + "\n" +
        "Altman Z={} ({})\n".format(Z, zone) +
        "Beneish M={} ({})\n".format(M, m_lbl) +
        (graham_str + "\n" if graham_str else "") +
        "News: {} (net={:+d}, pos={}, neg={})\n".format(news_label, news_score, pos_c, neg_c) +
        cagr_str + "\n\n"
        "===INSIGHTS===\n"
        "Write:\n**Executive Summary** (3 bullets, cite a number each)\n"
        "**Financial Health** (profitability, debt, cashflow — 3 bullets)\n"
        "**News & Sentiment Impact** (2 bullets)\n"
        "**Key Risks** (2 bullets)\n"
        "**Recommendation**: BUY/HOLD/SELL — one sentence.\n"
        "TAGS: Classify: Earnings(N), M&A(N), Regulatory(N), ESG(N), Analyst(N), Other(N)\n\n"
        "===PEER===\n"
        "Table: | Metric | {} | Industry Avg | vs Peers |\n".format(ticker) +
        "Include: Gross Margin, Net Margin, ROE, ROA, D/E, P/E.\n"
        "vs Peers: above avg / in-line / below avg. Then 2 sentences on positioning.\n\n"
        "===FORECAST===\n"
        "3-year table: | Year | Revenue ({}) | Growth | Net Income ({}) | Net Margin |\n".format(unit, unit) +
        "Then Bear/Base/Bull scenario table.\n"
        "Key assumptions (2 bullets). Confidence: High/Medium/Low."
    )

    raw = _gemini_call(prompt, timeout=30)

    if raw:
        # Parse the 3 sections
        def _extract(text, marker_start, marker_end=None):
            tag = f"==={marker_start}==="
            start = text.find(tag)
            if start == -1: return None
            start += len(tag)
            if marker_end:
                end = text.find(f"==={marker_end}===")
                return text[start:end].strip() if end > start else text[start:].strip()
            return text[start:].strip()

        insights_raw  = _extract(raw, "INSIGHTS", "PEER")
        peer_raw      = _extract(raw, "PEER",     "FORECAST")
        forecast_raw  = _extract(raw, "FORECAST")

        # UPGRADE 8 parse block
        global _news_tags_global
        if insights_raw:
            tags_m = _re.search(r'TAGS:(.*)', insights_raw)
            if tags_m:
                _news_tags_global = '**News Topic Tags:** ' + tags_m.group(1).strip()
                insights_raw = _re.sub(r'TAGS:.*', '', insights_raw).strip()
            else: _news_tags_global = ''
        insights = (
            '## AI Financial Analysis\n\n' + insights_raw + '\n\n---\n\n'
            + news_data.get('summary_md', '')
            if insights_raw else None
        )
        peer     = peer_raw
        forecast = forecast_raw
    else:
        insights = peer = forecast = None
        _news_tags_global = ''

    # Rule-based fallback for any missing section
    if not insights:
        insights = (_rule_based_insights(df, ratios, Z, zone, M, m_lbl)
                    + "\n\n---\n\n" + news_data.get("summary_md",""))
    if not peer:
        peer = _rule_based_peer(ratios)
    if not forecast:
        forecast = _rule_based_forecast(df)

    return insights, peer, forecast


def generate_sentiment_scores(df, ratios, Z, M):
    scores = {}
    if "Revenue" in df.columns and len(df) > 2:
        rg = df["Revenue"].pct_change().tail(3).mean()
        scores["Revenue Trend"] = ("BULLISH",C["teal"],90) if rg>0.10 else (("NEUTRAL",C["amber"],55) if rg>0.02 else ("BEARISH",C["red"],20))
    nm  = ratios.get("Net Margin %",0) or 0
    scores["Profitability"] = ("BULLISH",C["teal"],85) if nm>12 else (("NEUTRAL",C["amber"],50) if nm>4 else ("BEARISH",C["red"],15))
    de  = ratios.get("Debt-to-Equity",999) or 999
    scores["Leverage"] = ("BULLISH",C["teal"],88) if de<0.5 else (("NEUTRAL",C["amber"],52) if de<1.5 else ("BEARISH",C["red"],18))
    cfq = ratios.get("Cash Flow Quality",0) or 0
    scores["Cash Flow"] = ("BULLISH",C["teal"],82) if cfq>1.0 else (("NEUTRAL",C["amber"],48) if cfq>0.6 else ("BEARISH",C["red"],12))
    if Z is not None:
        scores["Bankruptcy Risk"] = ("SAFE",C["teal"],92) if Z>2.99 else (("WATCH",C["amber"],45) if Z>1.81 else ("DANGER",C["red"],10))
    if M is not None:
        scores["Earnings Quality"] = ("CLEAN",C["teal"],88) if M<-2.22 else ("SUSPECT",C["red"],15)
    return scores


def viz_sentiment_dashboard(scores):
    if not scores or len(scores) < 2: return None
    labels=[k for k in scores]; sentiments=[scores[k][0] for k in labels]
    colors=[scores[k][1] for k in labels]; values=[scores[k][2] for k in labels]
    fig = make_subplots(rows=1,cols=2,column_widths=[0.5,0.5],
        subplot_titles=["<b>Sentiment Scores</b>","<b>Signal Strength Radar</b>"],
        specs=[[{"type":"xy"},{"type":"polar"}]])
    fig.add_trace(go.Bar(y=labels,x=values,orientation="h",marker_color=colors,marker_line_width=0,
        text=sentiments,textposition="inside",textfont=dict(size=12,color=C["text"],family="Courier New"),
        showlegend=False),row=1,col=1)
    fig.update_xaxes(range=[0,100],row=1,col=1)
    theta=labels+[labels[0]]; r=values+[values[0]]
    fig.add_trace(go.Scatterpolar(r=r,theta=theta,fill="toself",fillcolor="rgba(77,159,255,0.15)",
        line=dict(color=C["blue"],width=2),marker=dict(size=8,color=C["blue"]),showlegend=False),row=1,col=2)
    base_layout(fig,"Financial Sentiment Dashboard",440,"Multi-dimensional sentiment scoring")
    fig.update_layout(polar=dict(bgcolor=C["panel"],
        radialaxis=dict(visible=True,range=[0,100],tickfont=dict(color=C["muted"],size=9),gridcolor=C["dim"]),
        angularaxis=dict(tickfont=dict(color=C["text"],size=10),gridcolor=C["dim"])))
    return wm(fig)


def grounded_qa(df, question):
    ctx = df.to_string(index=False)
    prompt = f"""Financial analyst. Answer using ONLY this data (max 150 words, cite numbers).
Data:
{ctx}
Question: {question}"""
    result = _gemini_call(prompt, timeout=20)
    if result:
        return result
    # Rule-based QA fallback
    q = question.lower()
    if "revenue" in q:
        if "Revenue" in df.columns:
            return f"Latest revenue: {df['Revenue'].iloc[-1]:,.0f}M. History: {df[['Year','Revenue']].to_string(index=False)}"
    if "profit" in q or "income" in q:
        if "Net_Income" in df.columns:
            return f"Latest net income: {df['Net_Income'].iloc[-1]:,.0f}M. History: {df[['Year','Net_Income']].to_string(index=False)}"
    if "debt" in q:
        if "Total_Debt" in df.columns:
            return f"Latest total debt: {df['Total_Debt'].iloc[-1]:,.0f}M."
    return f"Available data columns: {', '.join(df.columns.tolist())}. Please ask about a specific metric."


# UPGRADE 3c — News sentiment bar chart
def viz_news_sentiment_bar(news_data):
    pos = news_data.get("pos_count", 0)
    neg = news_data.get("neg_count", 0)
    neu = news_data.get("neu_count", 0)
    if pos + neg + neu == 0: return None
    fig = go.Figure(go.Bar(
        x=["Positive", "Neutral", "Negative"], y=[pos, neu, neg],
        marker_color=[C["teal"], C["amber"], C["red"]],
        marker_line_width=0, opacity=0.9,
        text=[pos, neu, neg], textposition="auto",
        textfont=dict(color=C["text"], size=14),
        hovertemplate="<b>%{x}</b><br>%{y} articles<extra></extra>"
    ))
    base_layout(fig, "News Sentiment Distribution", 300,
        "Overall: {}".format(news_data.get("label", "—")))
    return wm(fig)

print("AI Engine v5.0 ready — enhanced prompt, news cards, topic tags, Bear/Base/Bull")


AI Engine v5.0 ready — enhanced prompt, news cards, topic tags, Bear/Base/Bull


In [13]:
# Cell 9 — Export to Excel (Upgrade 7)
# UPGRADE 7 — Export to Excel (5-sheet workbook)
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

def export_to_excel_file():
    if _df is None: return None
    ticker_s = (_ticker or "export").replace(".", "_")
    path = "/tmp/FinIQ_v5_{}.xlsx".format(ticker_s)
    wb = Workbook()

    def style_header(ws, row=1):
        for cell in ws[row]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="0A0E1A")
            cell.alignment = Alignment(horizontal="center")

    def auto_width(ws):
        for col in ws.columns:
            max_len = max((len(str(cell.value or "")) for cell in col), default=10)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

    def alt_rows(ws, start=2, c1="EEF2FF", c2="FFFFFF"):
        for i, row in enumerate(ws.iter_rows(min_row=start)):
            fill = PatternFill("solid", fgColor=c1 if i % 2 == 0 else c2)
            for cell in row: cell.fill = fill

    # Sheet 1: Summary
    ws1 = wb.active
    ws1.title = "Summary"
    ci = _company_info
    ws1.append(["Field", "Value"])
    style_header(ws1)
    rows = [
        ("Company",    ci.get("company_name", "")),
        ("Ticker",     ci.get("ticker", "")),
        ("Currency",   ci.get("currency", "")),
        ("Unit",       ci.get("unit", "")),
        ("Source",     ci.get("source", "")),
        ("Years",      ", ".join(_df["Year"].astype(str).tolist())),
        ("", ""),
        ("KEY RATIOS", ""),
    ]
    for k, v in _ratios.items():
        if v is not None: rows.append((k, str(v)))
    rows += [
        ("Altman Z-Score",  str(_risk.get("Z", ""))),
        ("Altman Zone",     str(_risk.get("zone", ""))),
        ("Beneish M-Score", str(_risk.get("M", ""))),
        ("Beneish Label",   str(_risk.get("m_lbl", ""))),
    ]
    for row in rows: ws1.append(row)
    alt_rows(ws1)
    auto_width(ws1)

    # Sheet 2: Annual Financials
    ws2 = wb.create_sheet("Annual Financials")
    cols = list(_df.columns)
    ws2.append(cols)
    style_header(ws2)
    for _, row in _df.iterrows():
        ws2.append([row[c] for c in cols])
    alt_rows(ws2)
    auto_width(ws2)

    # Sheets 3-5: Raw Statements
    try:
        stmts = fetch_statements(_ticker, "Annual") if _ticker and _ticker != "UPLOAD" else {}
        for sheet_name, key, priority in [
            ("Income Statement", "income",   INCOME_PRIORITY),
            ("Balance Sheet",    "balance",  BALANCE_PRIORITY),
            ("Cash Flow",        "cashflow", CF_PRIORITY),
        ]:
            ws = wb.create_sheet(sheet_name)
            raw_df = stmts.get(key)
            if raw_df is not None and not raw_df.empty:
                headers = ["Metric"] + [str(c)[:10] for c in raw_df.columns]
                ws.append(headers)
                style_header(ws)
                ordered = []
                seen = set()
                for k in priority:
                    if k in raw_df.index and k not in seen:
                        ordered.append(k); seen.add(k)
                for k in raw_df.index:
                    if k not in seen: ordered.append(k); seen.add(k)
                for idx in ordered[:20]:
                    label = STMT_ROW_LABELS.get(idx, idx)
                    vals = [label]
                    for v in raw_df.loc[idx].tolist():
                        try:
                            fv = float(v)
                            vals.append(round(fv/1e6, 2) if str(v) not in ("nan","None","<NA>") else None)
                        except: vals.append(None)
                    ws.append(vals)
                alt_rows(ws)
            else:
                ws.append(["No data available"])
            auto_width(ws)
    except Exception as e:
        print("Statement export error: {}".format(e))

    wb.save(path)
    print("Exported to {}".format(path))
    return path

print("Export engine ready (Upgrade 7).")


Export engine ready (Upgrade 7).


In [15]:
# ════════════════════════════════════════════════════════════════════════════
# FinIQ v5 — Cell 10  (Gradio UI — App-Ready Rewrite)
# Runs DIRECTLY inside the notebook. All backend from Cells 1-9 in scope.
# ════════════════════════════════════════════════════════════════════════════

import gradio as gr
import warnings, os
warnings.filterwarnings("ignore")

_df           = None
_ticker       = ""
_ratios       = {}
_risk         = {}
_company_info = {}
_info         = {}
_news_data    = {}
_insights     = ""
_peer         = ""
_forecast     = ""
_watchlist    = []


def _set_state(df, ticker, ratios, risk, ci, info):
    global _df, _ticker, _ratios, _risk, _company_info, _info
    _df, _ticker, _ratios, _risk, _company_info, _info = df, ticker, ratios, risk, ci, info

def _do_risk(df, info=None):
    try:    r = compute_ratios(df, info or {})
    except: r = {}
    try:    Z, zone, zcol = altman_z(df)
    except: Z, zone, zcol = None, "N/A", "#6B84A3"
    try:    M, m_lbl, _  = beneish_m(df)
    except: M, m_lbl     = None, "N/A"
    return r, dict(Z=Z, zone=zone, zcol=zcol, M=M, m_lbl=m_lbl)

def _safe(fn, *args):
    try:    return fn(*args)
    except: return None

def _kpi_row(ratios, risk):
    Z, zone  = risk.get("Z"), risk.get("zone", "N/A")
    M, m_lbl = risk.get("M"), risk.get("m_lbl", "N/A")
    zone_icon = "🟢" if "Safe" in str(zone) else ("🔴" if "Distress" in str(zone) else "🟡")
    m_icon    = "🟢" if "Low" in str(m_lbl) else "🔴"
    lines = ["| Metric | Value | | Metric | Value |", "|--------|-------|---|--------|-------|"]
    items = [(k, v) for k, v in ratios.items() if v is not None]
    for i in range(0, len(items), 2):
        k1, v1 = items[i]
        if i + 1 < len(items):
            k2, v2 = items[i+1]
            lines.append(f"| **{k1}** | `{v1}` | | **{k2}** | `{v2}` |")
        else:
            lines.append(f"| **{k1}** | `{v1}` | | | |")
    lines.append(f"| **Altman Z-Score** | `{Z}` {zone_icon} *{zone}* | | **Beneish M-Score** | `{M}` {m_icon} *{m_lbl}* |")
    return "\n".join(lines)


def _build_figs(df, ratios, risk, ticker, info):
    Z, zone, zcol = risk.get("Z"), risk.get("zone","N/A"), risk.get("zcol","#6B84A3")
    M, m_lbl      = risk.get("M"), risk.get("m_lbl","N/A")
    return [
        _safe(viz_revenue,      df),
        _safe(viz_margins,      df),
        _safe(viz_eps_trend,    df, info),
        _safe(viz_waterfall,    df),
        _safe(viz_balance_sheet,df),
        _safe(viz_cashflow,     df),
        _safe(viz_dupont,       df),
        _safe(viz_dcf_heatmap,  df),
        _safe(viz_ratio_gauges, ratios),
        _safe(viz_altman_gauge, Z, zone, zcol),
        _safe(viz_beneish,      df),
        _safe(viz_red_flags,    df, ratios, Z, zone, M, m_lbl),
        _safe(viz_price_history, ticker) if ticker != "UPLOAD" else None,
        _safe(viz_sentiment_dashboard, generate_sentiment_scores(df, ratios, Z, M)),
    ]

def on_fetch(query):
    global _watchlist
    blank = [None]*14
    if not query or not query.strip():
        return ("⚠️ Enter a company name or ticker.", "", "", "", *blank, "", "")
    try:    raw = fetch_financials_from_web(query.strip())
    except Exception as e:
        return (f"❌ {e}", "", "", "", *blank, "", "")
    ci     = {k: raw.get(k,"") for k in ["company_name","ticker","currency","unit","source"]}
    ticker = raw.get("ticker","")
    info   = raw.get("info",{})
    df     = json_to_dataframe(raw)
    ratios, risk = _do_risk(df, info)
    _set_state(df, ticker, ratios, risk, ci, info)
    if ticker and ticker not in ("UPLOAD","") and ticker not in _watchlist:
        _watchlist.append(ticker); _watchlist = _watchlist[-5:]
    try:    header = build_company_header(info, ratios)
    except: header = f"## {ci['company_name']} ({ticker})"
    years_str = ", ".join(df["Year"].astype(str).tolist())
    status    = f"✅  **{ci['company_name']}** ({ticker})  ·  {ci.get('currency','')}  ·  Years: {years_str}"
    figs      = _build_figs(df, ratios, risk, ticker, info)
    peer_md   = _rule_based_peer(ratios)
    fore_md   = _rule_based_forecast(df)
    return (status, header, _kpi_row(ratios, risk), peer_md, *figs, peer_md, fore_md)

def on_excel(file):
    blank = [None]*14
    if file is None:
        return ("⚠️ No file uploaded.", "", "", "", *blank, "", "")
    try:    df = load_excel_and_normalize(file.name)
    except Exception as e:
        return (f"❌ {e}", "", "", "", *blank, "", "")
    ci = {"company_name":"Uploaded","ticker":"UPLOAD","currency":"","unit":"millions","source":"Excel"}
    ratios, risk = _do_risk(df)
    _set_state(df, "UPLOAD", ratios, risk, ci, {})
    years_str = ", ".join(df["Year"].astype(str).tolist())
    figs      = _build_figs(df, ratios, risk, "UPLOAD", {})
    peer_md   = _rule_based_peer(ratios)
    fore_md   = _rule_based_forecast(df)
    return ("✅  Excel loaded  ·  Years: " + years_str,
            "## Uploaded Financial Data",
            _kpi_row(ratios, risk), peer_md, *figs, peer_md, fore_md)

def on_reset():
    global _df, _ticker, _ratios, _risk, _company_info, _info, _news_data, _insights, _peer, _forecast
    _df = None; _ticker = ""
    _ratios = _risk = _company_info = _info = _news_data = {}
    _insights = _peer = _forecast = ""
    blank = [None]*14
    return ("", "", "", "", *blank, "", "")

def _raw_df_to_table(raw_df, priority, max_rows):
    import pandas as pd
    if raw_df is None or raw_df.empty:
        return pd.DataFrame({"Info": ["No data available"]})
    ordered, seen = [], set()
    for k in priority:
        if k in raw_df.index and k not in seen:
            ordered.append(k); seen.add(k)
    for k in raw_df.index:
        if k not in seen:
            ordered.append(k); seen.add(k)
    ordered = ordered[:int(max_rows)]
    cols = list(raw_df.columns)
    col_labels = []
    for c in cols:
        try:    col_labels.append(pd.Timestamp(c).strftime("%b %Y"))
        except: col_labels.append(str(c)[:10])
    rows = []
    for key in ordered:
        label = STMT_ROW_LABELS.get(key, key)
        vals  = raw_df.loc[key].tolist()
        fmt   = []
        for v in vals:
            try:
                fv = float(v)
                if pd.isna(fv):        fmt.append("—")
                elif abs(fv) >= 1e9:   fmt.append(f"{fv/1e9:,.2f}B")
                elif abs(fv) >= 1e6:   fmt.append(f"{fv/1e6:,.0f}M")
                elif abs(fv) >= 1e3:   fmt.append(f"{fv/1e3:,.1f}K")
                else:                  fmt.append(f"{fv:,.2f}")
            except:
                s = str(v)
                fmt.append("—" if s in ("nan","None","<NA>","") else s)
        yoy = "—"
        try:
            c_val = float(vals[-1]); p_val = float(vals[-2])
            if not (pd.isna(c_val) or pd.isna(p_val)) and abs(p_val) > 1e-9:
                pct = (c_val - p_val) / abs(p_val) * 100
                yoy = f"{'▲' if pct >= 0 else '▼'} {abs(pct):.1f}%"
        except: pass
        rows.append([label] + fmt + [yoy])
    out_cols = ["Metric"] + col_labels + ["YoY"]
    return pd.DataFrame(rows, columns=out_cols)

def _stmt_insight(raw_df, kind):
    if raw_df is None or raw_df.empty:
        return ""
    import pandas as pd
    lines = []
    try:
        if kind == "income":
            rev  = raw_df.loc["Total Revenue"] if "Total Revenue" in raw_df.index else None
            ni   = raw_df.loc["Net Income"]    if "Net Income"    in raw_df.index else None
            ebit = raw_df.loc["EBIT"]          if "EBIT"          in raw_df.index else None
            if rev is not None:
                r_vals = [float(v) for v in rev.tolist() if str(v) not in ("nan","None","<NA>")]
                if len(r_vals) >= 2:
                    cagr = ((r_vals[-1]/r_vals[0])**(1/max(len(r_vals)-1,1))-1)*100
                    trend = "growing" if cagr > 3 else ("declining" if cagr < -2 else "stable")
                    lines.append(f"**Revenue** is {trend} at **{cagr:+.1f}% CAGR**.")
            if ni is not None:
                ni_vals = [float(v) for v in ni.tolist() if str(v) not in ("nan","None","<NA>")]
                if len(ni_vals) >= 2 and ni_vals[-2] != 0:
                    ni_chg = (ni_vals[-1]-ni_vals[-2])/abs(ni_vals[-2])*100
                    lines.append(f"**Net Income** changed **{ni_chg:+.1f}%** YoY (latest: {ni_vals[-1]/1e9:.2f}B).")
            if ebit is not None:
                eb_vals = [float(v) for v in ebit.tolist() if str(v) not in ("nan","None","<NA>")]
                if eb_vals:
                    lines.append(f"**EBIT** latest: {eb_vals[-1]/1e9:.2f}B.")
        elif kind == "balance":
            ta = raw_df.loc["Total Assets"]        if "Total Assets"        in raw_df.index else None
            td = raw_df.loc["Total Debt"]          if "Total Debt"          in raw_df.index else None
            eq = raw_df.loc["Stockholders Equity"] if "Stockholders Equity" in raw_df.index else (
                 raw_df.loc["Common Stock Equity"] if "Common Stock Equity" in raw_df.index else None)
            if ta is not None:
                ta_v = [float(v) for v in ta.tolist() if str(v) not in ("nan","None","<NA>")]
                if ta_v: lines.append(f"**Total Assets**: {ta_v[-1]/1e9:.2f}B.")
            if td is not None and eq is not None:
                td_v = [float(v) for v in td.tolist() if str(v) not in ("nan","None","<NA>")]
                eq_v = [float(v) for v in eq.tolist() if str(v) not in ("nan","None","<NA>")]
                if td_v and eq_v and eq_v[-1] != 0:
                    de  = td_v[-1]/eq_v[-1]
                    lev = "conservative" if de < 0.5 else ("moderate" if de < 1.5 else "high")
                    lines.append(f"**D/E Ratio**: {de:.2f}x ({lev} leverage).")
        elif kind == "cashflow":
            ocf = raw_df.loc["Operating Cash Flow"] if "Operating Cash Flow" in raw_df.index else None
            fcf = raw_df.loc["Free Cash Flow"]      if "Free Cash Flow"      in raw_df.index else None
            if ocf is not None:
                ocf_v = [float(v) for v in ocf.tolist() if str(v) not in ("nan","None","<NA>")]
                if ocf_v: lines.append(f"**Operating CF**: {ocf_v[-1]/1e9:.2f}B.")
            if fcf is not None:
                fcf_v = [float(v) for v in fcf.tolist() if str(v) not in ("nan","None","<NA>")]
                if fcf_v:
                    q = "positive ✅" if fcf_v[-1] > 0 else "negative ⚠️"
                    lines.append(f"**Free Cash Flow**: {fcf_v[-1]/1e9:.2f}B — {q}.")
    except Exception:
        pass
    return "  \n".join(lines)

def on_statements(period, max_rows):
    import pandas as pd
    empty = pd.DataFrame({"Info": ["—"]})
    if _df is None:
        return empty, empty, empty, "⚠️ Fetch data first.", "", "", ""
    if not _ticker or _ticker == "UPLOAD":
        return empty, empty, empty, "⚠️ Statements for live tickers only.", "", "", ""
    try:
        result = fetch_statements(_ticker, period)
        if result.get("error"):
            return empty, empty, empty, f"❌ {result['error']}", "", "", ""
        inc_raw = result.get("income")
        bal_raw = result.get("balance")
        cf_raw  = result.get("cashflow")
        inc_df  = _raw_df_to_table(inc_raw, INCOME_PRIORITY,  max_rows)
        bal_df  = _raw_df_to_table(bal_raw, BALANCE_PRIORITY, max_rows)
        cf_df   = _raw_df_to_table(cf_raw,  CF_PRIORITY,      max_rows)
        inc_ins = _stmt_insight(inc_raw, "income")
        bal_ins = _stmt_insight(bal_raw, "balance")
        cf_ins  = _stmt_insight(cf_raw,  "cashflow")
        status  = f"✅  Loaded {period} · {len(inc_df)} income rows · {len(bal_df)} balance rows · {len(cf_df)} CF rows"
        return inc_df, bal_df, cf_df, status, inc_ins, bal_ins, cf_ins
    except Exception as e:
        return empty, empty, empty, f"❌ {e}", "", "", ""

def on_ai():
    global _news_data, _insights, _peer, _forecast
    if _df is None: return "⚠️ Fetch data first.", "", ""
    Z, zone  = _risk.get("Z"), _risk.get("zone","N/A")
    M, m_lbl = _risk.get("M"), _risk.get("m_lbl","N/A")
    if _ticker and _ticker != "UPLOAD":
        try:    _news_data = analyze_news_sentiment(_ticker)
        except: _news_data = {"summary_md":"*News fetch failed.*","cards_md":"",
                              "label":"N/A","score":0,"pos_count":0,"neg_count":0,"neu_count":0}
    else:
        _news_data = {"summary_md":"*No news for Excel uploads.*","cards_md":"",
                      "label":"N/A","score":0,"pos_count":0,"neg_count":0,"neu_count":0}
    try:
        ins, peer, fore = run_single_ai_call(
            _df, _ratios, Z, zone, M, m_lbl, _news_data,
            company_name=_company_info.get("company_name",""),
            ticker=_company_info.get("ticker",""),
            currency=_company_info.get("currency",""),
            unit=_company_info.get("unit","millions"),
        )
    except Exception:
        ins  = _rule_based_insights(_df, _ratios, Z, zone, M, m_lbl)
        peer = _rule_based_peer(_ratios)
        fore = _rule_based_forecast(_df)
    _insights, _peer, _forecast = ins, peer, fore
    return ins, peer, fore

def on_news():
    global _news_data
    if not _ticker or _ticker == "UPLOAD":
        return "*News available for live tickers only.*", None, ""
    try:    _news_data = analyze_news_sentiment(_ticker)
    except Exception as e: return f"❌ {e}", None, ""
    chart   = _safe(viz_news_sentiment_bar, _news_data)
    summary = (f"**{_news_data.get('label','—')}**  ·  "
               f"Net: **{_news_data.get('score',0):+d}**  ·  "
               f"🟢 {_news_data.get('pos_count',0)} positive  "
               f"🔴 {_news_data.get('neg_count',0)} negative  "
               f"⚪ {_news_data.get('neu_count',0)} neutral")
    cards = _news_data.get("cards_md") or _news_data.get("summary_md","")
    return cards, chart, summary

def on_export():
    if _df is None: return "⚠️ Fetch data first."
    try:
        path = export_to_excel_file()
        if path and os.path.exists(path):
            from google.colab import files
            files.download(path)
            return f"✅  Downloading: {os.path.basename(path)}"
        return "❌ Export produced no file."
    except Exception as e:
        return f"❌ {e}"

def on_qa(question):
    if _df is None:                          return "⚠️ Fetch data first."
    if not question or not question.strip(): return "⚠️ Enter a question."
    try:    return grounded_qa(_df, question.strip())
    except Exception as e: return f"❌ {e}"


CSS = (
    "body,.gradio-container{"
    "background:#0A0E1A !important;color:#E8EEF7 !important;"
    "font-family:'JetBrains Mono','Courier New',monospace !important;}"
    ".gr-block,.gr-box,.gr-padded,.gr-panel,.gradio-accordion{"
    "background:#0F1624 !important;border:1px solid #1E2D45 !important;}"
    "textarea,input[type=text],input[type=number]{"
    "background:#141D2E !important;color:#E8EEF7 !important;"
    "border:1px solid #2A3F5F !important;border-radius:6px !important;}"
    "button.primary,button[variant=primary],.gr-button-primary{"
    "background:#4D9FFF !important;color:#0A0E1A !important;"
    "font-weight:700 !important;border:none !important;border-radius:6px !important;}"
    "button.secondary,.gr-button-secondary{"
    "background:#141D2E !important;color:#E8EEF7 !important;"
    "border:1px solid #2A3F5F !important;border-radius:6px !important;}"
    ".tab-nav button{background:#0F1624 !important;color:#6B84A3 !important;"
    "border-radius:6px 6px 0 0 !important;font-size:13px !important;padding:8px 16px !important;}"
    ".tab-nav button.selected{background:#1E2D45 !important;color:#4D9FFF !important;"
    "border-bottom:2px solid #4D9FFF !important;}"
    ".prose,.prose p,.prose li,.prose td,.prose th{color:#E8EEF7 !important;}"
    "table{width:100% !important;border-collapse:collapse !important;}"
    "th{background:#1E2D45 !important;color:#4D9FFF !important;padding:8px 12px !important;"
    "text-align:left !important;font-size:12px !important;}"
    "td{padding:6px 12px !important;border-bottom:1px solid #1E2D45 !important;"
    "font-size:12px !important;color:#E8EEF7 !important;}"
    "tr:hover td{background:#141D2E !important;}"
    "select,.gr-dropdown{background:#141D2E !important;color:#E8EEF7 !important;"
    "border:1px solid #2A3F5F !important;border-radius:6px !important;}"
    "footer{display:none !important;}"
    "h1,h2,h3,h4{color:#E8EEF7 !important;}"
)

with gr.Blocks(css=CSS, title="FinIQ v5") as demo:

    gr.Markdown(
        "# 📊  FinIQ v5  —  AI Financial Intelligence\n"
        "Live Data · 13 Charts · Statements · NLP News · AI Insights · Peer · Forecast · Q&A\n\n---"
    )

    with gr.Row(equal_height=True):
        inp_query = gr.Textbox(
            label="🔍  Company / Ticker",
            placeholder="Apple  |  TCS  |  RELIANCE.NS  |  MSFT  |  HDFC Bank",
            scale=6, lines=1, container=True)
        btn_fetch = gr.Button("🚀  Fetch & Analyse", variant="primary", scale=1, min_width=160)
        btn_reset = gr.Button("🔄  Reset",           variant="secondary", scale=1, min_width=90)

    with gr.Accordion("📁  Upload Excel instead", open=False):
        with gr.Row():
            inp_excel = gr.File(label="Upload .xlsx / .xls",
                                file_types=[".xlsx",".xls"], scale=4)
            btn_excel = gr.Button("📊  Analyse Excel", variant="secondary", scale=1, min_width=160)

    out_status = gr.Markdown(value="*Enter a ticker above and click Fetch & Analyse to begin.*")
    out_header = gr.Markdown(value="")

    with gr.Accordion("📐  Key Ratios & Risk Scores", open=True):
        out_kpi = gr.Markdown(value="*Fetch data to see ratios.*")

    gr.Markdown("---")

    with gr.Tabs():

        with gr.TabItem("📈  Charts"):
            gr.Markdown("### 📈  Financial Charts")
            with gr.Tabs():
                with gr.TabItem("💰  Revenue & Growth"):
                    gr.Markdown("Revenue trend, gross profit, and YoY growth rate.")
                    fig_rev = gr.Plot(show_label=False)
                with gr.TabItem("📊  Margins"):
                    gr.Markdown("Gross margin, EBITDA margin, and net margin over time.")
                    fig_mar = gr.Plot(show_label=False)
                with gr.TabItem("📉  EPS Trend"):
                    gr.Markdown("Earnings per share with trailing P/E reference line.")
                    fig_eps = gr.Plot(show_label=False)
                with gr.TabItem("🌊  Revenue Waterfall"):
                    gr.Markdown("Latest year: Revenue → COGS → Gross Profit → Net Income bridge.")
                    fig_wf  = gr.Plot(show_label=False)
                with gr.TabItem("🏦  Balance Sheet"):
                    gr.Markdown("Total assets, total debt, and equity over time.")
                    fig_bs  = gr.Plot(show_label=False)
                with gr.TabItem("💵  Cash Flow"):
                    gr.Markdown("Operating CF, free cash flow, and CapEx.")
                    fig_cf  = gr.Plot(show_label=False)
                with gr.TabItem("🔗  DuPont"):
                    gr.Markdown("ROE decomposition: margin × asset turnover × leverage.")
                    fig_dp  = gr.Plot(show_label=False)
                with gr.TabItem("🌡  DCF Sensitivity"):
                    gr.Markdown("EV/FCF multiple heatmap across growth vs discount rate.")
                    fig_dcf = gr.Plot(show_label=False)
                with gr.TabItem("🎯  Ratio Gauges"):
                    gr.Markdown("Key financial ratios as gauge charts.")
                    fig_rg  = gr.Plot(show_label=False)
                with gr.TabItem("⚠️  Altman Z-Score"):
                    gr.Markdown(
                        "**Altman Z-Score** — Bankruptcy risk model.\n\n"
                        "🟢 Z > 2.99 = Safe Zone  ·  🟡 1.81 – 2.99 = Grey Zone  ·  🔴 Z < 1.81 = Distress Zone"
                    )
                    fig_az  = gr.Plot(show_label=False)
                with gr.TabItem("🔍  Beneish M-Score"):
                    gr.Markdown(
                        "**Beneish M-Score** — Earnings manipulation detector.\n\n"
                        "🟢 M < -2.22 = Low manipulation risk  ·  🔴 M > -2.22 = Possible manipulation"
                    )
                    fig_bm  = gr.Plot(show_label=False)
                with gr.TabItem("🚩  Red Flags"):
                    gr.Markdown("Automated financial red flag scorecard by severity.")
                    fig_rf  = gr.Plot(show_label=False)
                with gr.TabItem("📅  Price History"):
                    gr.Markdown("1-year OHLC candlestick with SMA 50/200 and volume.")
                    fig_ph  = gr.Plot(show_label=False)
                with gr.TabItem("🧭  Sentiment Radar"):
                    gr.Markdown("Multi-dimensional financial sentiment radar and bar chart.")
                    fig_sd  = gr.Plot(show_label=False)

        with gr.TabItem("📋  Statements"):
            gr.Markdown("### 📋  Financial Statements")
            with gr.Row():
                rad_period = gr.Radio(
                    choices=["Annual","Quarterly","TTM"], value="Annual",
                    label="🗓  Period", scale=1)
                sld_rows   = gr.Slider(5, 30, step=5, value=15,
                                       label="🔢  Rows per statement", scale=2)
                btn_stmts  = gr.Button("📥  Load Statements", variant="primary",
                                       scale=1, min_width=160)
            out_stmt_status = gr.Markdown(value="")
            with gr.Tabs():
                with gr.TabItem("📑  Income Statement"):
                    out_inc_ins = gr.Markdown(value="")
                    out_inc = gr.Dataframe(label="Income Statement", wrap=True, interactive=False)
                with gr.TabItem("🏛  Balance Sheet"):
                    out_bal_ins = gr.Markdown(value="")
                    out_bal = gr.Dataframe(label="Balance Sheet", wrap=True, interactive=False)
                with gr.TabItem("💸  Cash Flow"):
                    out_cf_ins  = gr.Markdown(value="")
                    out_cf  = gr.Dataframe(label="Cash Flow Statement", wrap=True, interactive=False)
            btn_stmts.click(
                on_statements,
                inputs=[rad_period, sld_rows],
                outputs=[out_inc, out_bal, out_cf, out_stmt_status,
                         out_inc_ins, out_bal_ins, out_cf_ins],
            )

        with gr.TabItem("🤖  AI Insights"):
            gr.Markdown(
                "### 🤖  AI-Powered Analysis\n"
                "One Gemini 2.0 Flash call returns **Insights + Peer Benchmarking + 3-Year Forecast** "
                "simultaneously. Falls back to rule-based analysis instantly if unavailable."
            )
            btn_ai = gr.Button("🚀  Generate AI + NLP Analysis", variant="primary")
            out_ai = gr.Markdown(value="*Click the button above to generate AI analysis.*")

        with gr.TabItem("🏆  Peer"):
            gr.Markdown(
                "### 🏆  Peer Comparison & Benchmarking\n"
                "Rule-based benchmarks load instantly after Fetch. "
                "Run AI Analysis for Gemini-powered peer positioning."
            )
            out_peer = gr.Markdown(value="*Fetch data first.*")

        with gr.TabItem("📅  Forecast"):
            gr.Markdown(
                "### 📅  3-Year Forward Forecast\n"
                "Trend extrapolation loads instantly. "
                "Run AI Analysis for Bear / Base / Bull scenario forecast."
            )
            out_forecast = gr.Markdown(value="*Fetch data first.*")

        with gr.TabItem("📰  News"):
            gr.Markdown("### 📰  News Sentiment Analysis")
            btn_news         = gr.Button("📰  Load News Sentiment", variant="primary")
            out_news_summary = gr.Markdown(value="")
            with gr.Row():
                with gr.Column(scale=1):
                    plt_news = gr.Plot(label="Sentiment Distribution")
                with gr.Column(scale=2):
                    out_news_cards = gr.Markdown(
                        value="*Click Load News Sentiment after fetching a live ticker.*"
                    )
            btn_news.click(on_news, inputs=[],
                           outputs=[out_news_cards, plt_news, out_news_summary])

        # ── FIX: Export tab is now correctly INSIDE with gr.Tabs() ──
        with gr.TabItem("📥  Export"):
            gr.Markdown(
                "### 📥  Export to Excel\n"
                "5-sheet workbook: **Summary · Annual Financials · "
                "Income Statement · Balance Sheet · Cash Flow**\n\n"
                "*The file will download automatically via your browser.*"
            )
            btn_export     = gr.Button("📥  Generate & Download Excel", variant="primary")
            out_exp_status = gr.Markdown(value="")
            btn_export.click(on_export, inputs=[], outputs=[out_exp_status])

        with gr.TabItem("💬  Q&A"):
            gr.Markdown(
                "### 💬  Ask Anything About the Data\n"
                "Answers grounded strictly in loaded financial data. "
                "Gemini 2.0 Flash with rule-based fallback."
            )
            inp_qa = gr.Textbox(
                label="Your question",
                placeholder="Which year had the best margins?  |  Is debt rising?  |  What drives ROE?",
                lines=2)
            btn_qa = gr.Button("Ask", variant="primary")
            out_qa = gr.Markdown(value="")
            btn_qa.click(on_qa,  inputs=[inp_qa], outputs=[out_qa])
            inp_qa.submit(on_qa, inputs=[inp_qa], outputs=[out_qa])

    gr.Markdown(
        "\n\n---\n"
        "<center><sub>FinIQ v5  ·  yfinance (free)  ·  Gemini 2.0 Flash  ·  "
        "Rule-based fallback  ·  Graham Number  ·  Analyst Targets</sub></center>"
    )

    _all_figs   = [fig_rev, fig_mar, fig_eps, fig_wf,
                   fig_bs,  fig_cf,  fig_dp,  fig_dcf,
                   fig_rg,  fig_az,  fig_bm,  fig_rf,
                   fig_ph,  fig_sd]
    _fetch_outs = [out_status, out_header, out_kpi, out_peer,
                   *_all_figs, out_peer, out_forecast]

    btn_fetch.click(on_fetch, inputs=[inp_query], outputs=_fetch_outs)
    btn_excel.click(on_excel, inputs=[inp_excel], outputs=_fetch_outs)
    btn_reset.click(on_reset, inputs=[],          outputs=_fetch_outs)
    btn_ai.click(on_ai, inputs=[], outputs=[out_ai, out_peer, out_forecast])


print("✅ FinIQ v5 Gradio UI ready.")
demo.launch(
    share=False,
    debug=False,
    show_error=True,
    quiet=True,
    inbrowser=False,
)

✅ FinIQ v5 Gradio UI ready.


<IPython.core.display.Javascript object>